# Torun Reservoir WAQ — FPV Scenario Diagnostic (NetCDF-aware)

Point-monitor diagnostic at CAB and BIS stations, comparing baseline to
FPV scenario 02 (continuous coverage).

**Data sources (mixed):**
- **NetCDF** (`.map.nc`, from the conversion notebook): preferred for state
  variables that are in the .map output — OXY, NH4, NO3, PO4, Si, POC1, PON1,
  POP1, AAP, Opal, all algal ecotypes. Read at the CAB column's (M, N).
- **`.his`** (point-monitor binary): used for diagnostic outputs that are
  computed by BLOOM but not written to .map — Chlfa (BLOOM internal),
  SecchiDept, RadAve, VWind, Temp, ExtVl, Depth, Limit\*, BloomDepth, DayL,
  SatPercOXY, SedOXYDem.

**Why a mixed approach:** the .his .seg_id field is *not* a FLOW segment ID
(verified — for CAB(8) it returns segment 3, a boundary cell at K=0, not the
actual CAB water-column cell at K=7). The CAB (M, N) is therefore
auto-detected by matching NetCDF OXY against .his OXY at CAB(8) — the (M, N)
column whose K=7 OXY time series most closely matches the .his CAB(8) OXY
time series.

## Sections
1. Configuration and data loading
2. Auto-detect CAB column (M, N) by NetCDF ↔ .his matching
3. DO diagnostics
4. Nutrient diagnostics (depth-averaged)
5. Phytoplankton diagnostics
6. Seasonal difference boxplots
7. Performance summary table


## Section 1 — Configuration and loading

In [6]:
import os
import struct
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
BIN_DIR  = r'<MODEL_DIR>'
MAP_DIR  = os.path.join(BIN_DIR, 'map_files_scenarios')
HIS_DIR  = os.path.join(BIN_DIR, 'his_scenario_files')

RESULTS_DIR = os.path.join(BIN_DIR, 'results_scenarios')
os.makedirs(RESULTS_DIR, exist_ok=True)

# NetCDFs (one per scenario) — for state variables in .map
NC_BASE = os.path.join(MAP_DIR, 'Torun_WAQ_Apr2023.nc')
NC_FPV  = os.path.join(MAP_DIR, 'Torun_WAQ_Apr2023_scen02.nc')

# .his files — for diagnostic substances not in .map
HIS_BASE = os.path.join(HIS_DIR, 'Torun_WAQ_Apr2023.his')
HIS_FPV  = os.path.join(HIS_DIR, 'Torun_WAQ_Apr2023_scen02.his')

# ── Constants ────────────────────────────────────────────────────────────────
T0         = pd.Timestamp('2022-09-09')
SIM_START  = pd.Timestamp('2023-04-01')
SIM_END    = pd.Timestamp('2023-08-31')
ChlaCAlg   = 0.025

# Active WAQ layers and per-layer mid-depths
K_TOP = 7
K_BOT = 15

THICK_PCT_FLOW = [12.5, 6.25, 6.25, 6.25, 6.25, 6.25, 6.25, 6.25,
                  3.125, 3.125, 3.125, 3.125, 3.125, 3.125, 12.5, 12.5]
Z_TOTAL = 14.79
THICKNESS_M = {waq_k: THICK_PCT_FLOW[15 - waq_k] / 100.0 * Z_TOTAL
               for waq_k in range(16)}
# Cumulative mid-depth per active layer
LAYER_MID_DEPTHS = {}
cum = 0.0
for k in range(K_TOP, K_BOT + 1):
    LAYER_MID_DEPTHS[k] = cum + THICKNESS_M[k] / 2.0
    cum += THICKNESS_M[k]

# Interpolation weights for 0.7 m DO sensor (legacy)
w1, w2 = 0.30, 0.70

# .his K convention: 'CAB (8)' → WAQ K=7 (1-based labels in .his)
LAYER_BOUNDS_HIS = {  # legacy 1-based labels for .his
    8:(0,0.55), 9:(0.55,1.46), 10:(1.46,2.45),
    11:(2.45,3.40), 12:(3.40,4.45), 13:(4.45,5.80),
    14:(5.80,7.30), 15:(7.30,8.30)
}

print(f'Working directory: {BIN_DIR}')
print(f'Simulation window: {SIM_START.date()} → {SIM_END.date()}')

Working directory: <MODEL_DIR>
Simulation window: 2023-04-01 → 2023-08-31


In [7]:
# ── Load NetCDFs (state variables) ──────────────────────────────────────────
def open_scenario_window(path, t0=SIM_START, t1=SIM_END):
    ds = xr.open_dataset(path)
    return ds.sel(time=slice(t0, t1))

ds_B = open_scenario_window(NC_BASE)
ds_F = open_scenario_window(NC_FPV)
print(f'NetCDF Baseline: {len(ds_B.time)} timesteps, '
      f'{len(ds_B.data_vars)} substances')
print(f'NetCDF FPV:      {len(ds_F.time)} timesteps, '
      f'{len(ds_F.data_vars)} substances')
print(f'  Substances:    {list(ds_B.data_vars)}')

NetCDF Baseline: 609 timesteps, 25 substances
NetCDF FPV:      609 timesteps, 25 substances
  Substances:    ['Continuity', 'OXY', 'AAP', 'POC1', 'PON1', 'POP1', 'Opal', 'NH4', 'NO3', 'PO4', 'Si', 'FDIATOMS_E', 'FDIATOMS_P', 'FFLAGELA', 'GREENS_E', 'GREENS_N', 'GREENS_P', 'MICROCYS_E', 'MICROCYS_N', 'MICROCYS_P', 'AAPS1', 'DetCS1', 'DetNS1', 'DetPS1', 'DetSiS1']


In [8]:
# ── Load .his files (diagnostic outputs) ────────────────────────────────────
def load_his(his_path, t0=T0):
    """Read DELWAQ .his with (n_loc, n_sub) per-record layout."""
    with open(his_path, 'rb') as f:
        [f.read(40) for _ in range(4)]
        n_sub, n_loc = struct.unpack('<ii', f.read(8))
        sub_names = [f.read(20).decode('latin-1').strip() for _ in range(n_sub)]
        locations = []
        for _ in range(n_loc):
            _ = struct.unpack('<i', f.read(4))[0]
            name = f.read(20).decode('latin-1').strip()
            locations.append(name)

        header_size = f.tell()
        record_size = 4 + n_sub * n_loc * 4
        fsize = os.path.getsize(his_path)
        n_times = (fsize - header_size) // record_size

        data = np.empty((n_times, n_loc, n_sub), dtype=np.float32)
        timestamps = []
        f.seek(header_size)
        for i in range(n_times):
            ts = struct.unpack('<i', f.read(4))[0]
            data[i] = np.frombuffer(f.read(n_sub * n_loc * 4),
                                    dtype='<f4').reshape(n_loc, n_sub)
            timestamps.append(t0 + pd.Timedelta(seconds=ts))

    print(f'Loaded: {os.path.basename(his_path)} | '
          f'{n_sub} subs | {n_loc} locs | {n_times} steps')
    return {'data': data,
            'substances': sub_names,
            'idx': {s: i for i, s in enumerate(sub_names)},
            'locations': locations,
            'loc_idx': {l: i for i, l in enumerate(locations)},
            'timestamps': pd.DatetimeIndex(timestamps),
            'ts_index': pd.DatetimeIndex(timestamps)}


RB = load_his(HIS_BASE)
RF = load_his(HIS_FPV)

# Trim to window
for R in [RB, RF]:
    mask = (R['ts_index'] >= SIM_START) & (R['ts_index'] <= SIM_END)
    idx = np.where(mask)[0]
    R['data'] = R['data'][idx]
    R['timestamps'] = R['timestamps'][idx]
    R['ts_index']   = R['ts_index'][idx]

Loaded: Torun_WAQ_Apr2023.his | 42 subs | 20 locs | 1253 steps
Loaded: Torun_WAQ_Apr2023_scen02.his | 42 subs | 20 locs | 1253 steps


In [9]:
# ── CAB and BIS monitor lookups by 1-based K label ──────────────────────────
def find_monitor_idx(R, prefix, k_label):
    target = f'{prefix} ({k_label})'
    return R['loc_idx'].get(target)

# 1-based labels in .his — e.g. CAB (8) is WAQ K=7
cab_B = {k_label: find_monitor_idx(RB, 'CAB', k_label) for k_label in range(8, 17)}
cab_F = {k_label: find_monitor_idx(RF, 'CAB', k_label) for k_label in range(8, 17)}
bis_B = {k_label: find_monitor_idx(RB, 'BIS', k_label) for k_label in range(8, 17)}
bis_F = {k_label: find_monitor_idx(RF, 'BIS', k_label) for k_label in range(8, 17)}

# Drop any that weren't found
cab_B = {k: v for k, v in cab_B.items() if v is not None}
cab_F = {k: v for k, v in cab_F.items() if v is not None}
bis_B = {k: v for k, v in bis_B.items() if v is not None}
bis_F = {k: v for k, v in bis_F.items() if v is not None}

print('CAB monitor indices (.his):', cab_B)
print('BIS monitor indices (.his):', bis_B)
print(f'\n.his ↔ WAQ K mapping: CAB(N) = WAQ K=N-1, e.g. CAB(8)=K={K_TOP}')

CAB monitor indices (.his): {8: 2, 9: 3, 10: 4, 11: 5, 12: 6, 13: 7, 14: 8, 15: 9, 16: 10}
BIS monitor indices (.his): {8: 11, 9: 12, 10: 13, 11: 14, 12: 15, 13: 16, 14: 17, 15: 18, 16: 19}

.his ↔ WAQ K mapping: CAB(N) = WAQ K=N-1, e.g. CAB(8)=K=7


In [11]:
# ── Load segnum and new_to_old from the saved lookup ────────────────────
import numpy as np

LOOKUP_PATH = '<MODEL_DIR>/segment_lookup.npz'
lookup = np.load(LOOKUP_PATH)
segnum     = lookup['segnum']
new_to_old = lookup['new_to_old']
print(f'Loaded segnum (shape {segnum.shape}) and '
      f'new_to_old (length {len(new_to_old)}) from {LOOKUP_PATH}')

Loaded segnum (shape (60, 50, 16)) and new_to_old (length 10999) from <MODEL_DIR>/segment_lookup.npz


In [12]:
# ── FPV perimeter mask detection (data-driven from .bin radiation files) ──
# Identifies all (M, N) cells under the FPV array by finding cells where
# Baseline and FPV radiation differ at any sampled timestep. Self-contained.

# Radiation .bin file paths — adjust filenames if yours differ
RAD_BIN = {
    'Baseline': os.path.join(BIN_DIR, 'waq-sw_radiation_flux-noFPV_fixed.bin'),
    'FPV':      os.path.join(BIN_DIR, 'waq-daily-sw_radiation_flux-02towaq_fpv41_nogaps_fixed.bin'),
}

# Model dimensions — these come from Section 1 of the diagnostic notebook.
# Inlined here so this cell is self-contained.
if 'N_SEG' not in dir():
    N_SEG = 10999      # compact-segment count for the Torun WAQ model
if 'RAD_NT' not in dir():
    RAD_NT = 153       # number of timesteps in the .bin radiation files

def _detect_fpv_perimeter_mask(path_b, path_f, n_seg, sample_t=range(0, 200, 10)):
    """Return compact-segment boolean mask of cells under the FPV array."""
    rec_size = 4 + n_seg * 4
    differs_ever = np.zeros(n_seg, dtype=bool)
    with open(path_b, 'rb') as fb, open(path_f, 'rb') as ff:
        for t_idx in sample_t:
            fb.seek(t_idx * rec_size + 4)
            ff.seek(t_idx * rec_size + 4)
            row_b = np.frombuffer(fb.read(n_seg * 4), dtype='<f4').copy()
            row_f = np.frombuffer(ff.read(n_seg * 4), dtype='<f4').copy()
            differs_ever |= (np.abs(row_f - row_b) > 1e-6)
    return differs_ever


fpv_compact_mask = _detect_fpv_perimeter_mask(
    RAD_BIN['Baseline'], RAD_BIN['FPV'], N_SEG)
fpv_compact_idx  = np.where(fpv_compact_mask)[0]
print(f'FPV footprint: {fpv_compact_mask.sum()} of {N_SEG} compact segments '
      f'({fpv_compact_mask.sum()/N_SEG*100:.1f}%)')

# Reverse-map compact indices to (M, N) at the surface layer
fpv_mask_mn = np.zeros((segnum.shape[0], segnum.shape[1]), dtype=bool)
for compact_idx in fpv_compact_idx:
    flow_seg = new_to_old[compact_idx] + 1
    for m_ in range(segnum.shape[0]):
        found = False
        for n_ in range(segnum.shape[1]):
            v = segnum[m_, n_, K_TOP]
            if not np.isnan(v) and int(v) == flow_seg:
                fpv_mask_mn[m_, n_] = True
                found = True
                break
        if found:
            break

perim_da = xr.DataArray(fpv_mask_mn, dims=('M', 'N'),
                         coords={'M': np.arange(fpv_mask_mn.shape[0]),
                                 'N': np.arange(fpv_mask_mn.shape[1])})

print(f'FPV perimeter (M, N) cells: {fpv_mask_mn.sum()}')

FPV footprint: 622 of 10999 compact segments (5.7%)
FPV perimeter (M, N) cells: 469


## Section 2 — Auto-detect CAB column `(M, N)` in the NetCDF

The `.his` `seg_id` field doesn't map back to a FLOW segment, so we identify
CAB's grid position by matching: for each (M, N) in the NetCDF, compute the
correlation between the K=7 OXY time series and the `.his` CAB(8) OXY time
series. The best-matching `(M, N)` is CAB.

This is a one-off operation — once CAB's `(M, N)` is known, all subsequent
NetCDF queries use those coordinates.

In [13]:
def find_cab_mn(ds, his_run, his_loc_idx, substance='OXY'):
    """Find (M, N) in NetCDF whose K=K_TOP time series best matches .his at
    a target monitor. Returns (m, n, correlation)."""
    si = his_run['idx'][substance]
    his_ts = his_run['data'][:, his_loc_idx, si].astype(float)
    his_index = his_run['ts_index']

    # Common time grid: .his often has finer resolution than .map
    nc_da = ds[substance].sel(K=K_TOP)
    nc_times = pd.DatetimeIndex(nc_da.time.values)
    common = nc_times.intersection(his_index)
    if len(common) < 10:
        raise RuntimeError(f'Insufficient common timesteps ({len(common)}) '
                           f'between NetCDF and .his.')

    # Slice both to common times
    nc_aligned = nc_da.sel(time=common)
    his_aligned = pd.Series(his_ts, index=his_index).loc[common].values

    # For each (M, N) — compute correlation with .his
    arr = nc_aligned.values   # (T, M, N)
    his_aligned = his_aligned - np.nanmean(his_aligned)
    his_norm = np.sqrt(np.nansum(his_aligned ** 2))

    best_corr = -np.inf
    best_mn = (None, None)
    M_dim, N_dim = arr.shape[1], arr.shape[2]
    for mi in range(M_dim):
        for ni in range(N_dim):
            col = arr[:, mi, ni]
            if np.isnan(col).any() or col.std() < 1e-6:
                continue
            col_c = col - col.mean()
            denom = np.sqrt(np.nansum(col_c ** 2)) * his_norm
            if denom < 1e-12:
                continue
            corr = np.nansum(col_c * his_aligned) / denom
            if corr > best_corr:
                best_corr = corr
                best_mn = (mi, ni)
    return best_mn, best_corr


# Find CAB column by matching against Baseline OXY at CAB(8)
print('Auto-detecting CAB (M, N) by matching .his CAB(8) OXY against NetCDF...')
(cab_m, cab_n), corr = find_cab_mn(ds_B, RB, cab_B[8])
print(f'  CAB column → (M={cab_m}, N={cab_n})  (correlation={corr:.4f})')

# Same for BIS
print('Auto-detecting BIS (M, N)...')
(bis_m, bis_n), corr_b = find_cab_mn(ds_B, RB, bis_B[8])
print(f'  BIS column → (M={bis_m}, N={bis_n})  (correlation={corr_b:.4f})')

# Sanity check: at the CAB (M, N), all K=7..15 should have non-NaN data
print('\nCAB column non-NaN check across K (Baseline, first timestep):')
for k in range(K_TOP, K_BOT + 1):
    v = float(ds_B['OXY'].isel(time=0, M=cab_m, N=cab_n).sel(K=k).values)
    flag = 'wet' if not np.isnan(v) else 'dry'
    print(f'  K={k:2d}: OXY={v:.4f}  ({flag})')

Auto-detecting CAB (M, N) by matching .his CAB(8) OXY against NetCDF...
  CAB column → (M=23, N=36)  (correlation=1.0000)
Auto-detecting BIS (M, N)...
  BIS column → (M=37, N=12)  (correlation=1.0000)

CAB column non-NaN check across K (Baseline, first timestep):
  K= 7: OXY=10.8800  (wet)
  K= 8: OXY=10.8800  (wet)
  K= 9: OXY=10.8800  (wet)
  K=10: OXY=10.8800  (wet)
  K=11: OXY=10.8800  (wet)
  K=12: OXY=10.8800  (wet)
  K=13: OXY=10.8800  (wet)
  K=14: OXY=10.8800  (wet)
  K=15: OXY=10.8800  (wet)


## Section 3 — Unified data accessors (NetCDF preferred, `.his` fallback)

A single function returns a daily time series at the CAB column. Tries the
NetCDF first; if the substance isn't in the NetCDF, falls back to `.his` at
the corresponding monitor.

In [14]:
# Map .his 1-based K → WAQ 0-based K
def his_k_to_waq(k_label):
    return k_label - 1

def get_series(ds, his_run, substance, k_label, m=None, n=None, loc_dict=None):
    """Return a daily-resampled time series at CAB layer k_label.

    Priority:
    1. If substance is in the NetCDF (state variable), read ds[sub] at
       (M=m, N=n, K=k_label-1).
    2. Else fall back to .his at loc_dict[k_label].

    Args:
        ds:           xarray.Dataset for this scenario
        his_run:      .his dict for this scenario
        substance:    name (e.g. 'OXY', 'NH4', 'Chlfa')
        k_label:      .his-style 1-based K label (e.g. 8 = WAQ K=7)
        m, n:         CAB (M, N) — required if reading from NetCDF
        loc_dict:     .his monitor lookup dict (e.g. cab_B) — required for fallback
    """
    if substance in ds:
        waq_k = his_k_to_waq(k_label)
        if waq_k < K_TOP or waq_k > K_BOT:
            return pd.Series(dtype=float)
        da = ds[substance].isel(M=m, N=n).sel(K=waq_k)
        idx = pd.DatetimeIndex(da.time.values)
        ts = pd.Series(da.values, index=idx)
        return ts.resample('D').mean()

    # Fall back to .his
    if loc_dict is None or k_label not in loc_dict:
        return pd.Series(dtype=float)
    si = his_run['idx'].get(substance)
    if si is None:
        return pd.Series(dtype=float)
    li = loc_dict[k_label]
    ts = pd.Series(his_run['data'][:, li, si].astype(float),
                   index=his_run['ts_index'])
    return ts.resample('D').mean()


def get_depth_matrix(ds, his_run, substance, loc_dict, m=None, n=None,
                     max_depth=8.0):
    """Build a (n_times, n_layers) matrix at CAB across all active layers.

    Uses NetCDF where possible, .his fallback per substance. Returns
    (timestamps, depths, matrix)."""
    layers = [k_label for k_label in sorted(loc_dict.keys())
              if his_k_to_waq(k_label) in range(K_TOP, K_BOT + 1)]
    depths = [LAYER_MID_DEPTHS[his_k_to_waq(k_label)] for k_label in layers]

    cols = []
    for k_label in layers:
        s = get_series(ds, his_run, substance, k_label,
                       m=m, n=n, loc_dict=loc_dict)
        cols.append(s)
    if not cols:
        return None, None, None
    df = pd.concat(cols, axis=1)
    df.columns = layers
    return df.index, depths, df.values


def depth_avg(ds, his_run, substance, loc_dict, m=None, n=None, max_depth=8.0):
    """Thickness-weighted depth-averaged time series."""
    idx, depths, mat = get_depth_matrix(ds, his_run, substance, loc_dict,
                                         m=m, n=n, max_depth=max_depth)
    if mat is None:
        return pd.Series(dtype=float)
    weights = np.array([
        LAYER_BOUNDS_HIS.get(k_label, (0, 0))[1]
        - LAYER_BOUNDS_HIS.get(k_label, (0, 0))[0]
        for k_label in sorted(loc_dict.keys())
        if his_k_to_waq(k_label) in range(K_TOP, K_BOT + 1)
    ])
    weights = weights / weights.sum() if weights.sum() > 0 else weights
    weighted = (mat * weights[None, :]).sum(axis=1)
    return pd.Series(weighted, index=idx)


print('Data accessors loaded.')
print('  get_series(ds, his, "OXY", 8, m=cab_m, n=cab_n) → NetCDF')
print('  get_series(ds, his, "Chlfa", 8, m=cab_m, n=cab_n, loc_dict=cab_B) → .his')

Data accessors loaded.
  get_series(ds, his, "OXY", 8, m=cab_m, n=cab_n) → NetCDF
  get_series(ds, his, "Chlfa", 8, m=cab_m, n=cab_n, loc_dict=cab_B) → .his


## Section 4 — DO diagnostics

Read OXY from the NetCDF at CAB(M, N) across active layers. Compare baseline
to FPV at sensor depths.

In [16]:
xlim = (SIM_START, SIM_END)
xfmt = mdates.DateFormatter('%b\n%Y')
xloc = mdates.MonthLocator()

# DO depth profiles (Hovmöller)
ts_B, dep_B, mat_OB = get_depth_matrix(ds_B, RB, 'OXY', cab_B,
                                         m=cab_m, n=cab_n)
ts_F, dep_F, mat_OF = get_depth_matrix(ds_F, RF, 'OXY', cab_F,
                                         m=cab_m, n=cab_n)

OXY_LABEL = 'Dissolved Oxygen (g/m³)'

# Compute difference and locked colour ranges
common_t = pd.DatetimeIndex(ts_B).intersection(pd.DatetimeIndex(ts_F))
ib  = [i for i, t in enumerate(ts_B) if t in common_t]
ifs = [i for i, t in enumerate(ts_F) if t in common_t]
diff_mat = mat_OF[ifs] - mat_OB[ib]

# Absolute range: pooled 2-98% across both scenarios
all_abs = np.concatenate([
    mat_OB[~np.isnan(mat_OB)],
    mat_OF[~np.isnan(mat_OF)],
])
vmin, vmax = np.nanpercentile(all_abs, [2, 98])

# Difference range: symmetric 95%
dlim = np.nanpercentile(np.abs(diff_mat), 95)

# Colormaps
CMAP_ABS  = 'viridis'   # sequential — for absolute DO concentration
CMAP_DIFF = 'RdBu_r'    # diverging — for differences

fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharey = True)

# Absolute panels
panel_data_abs = [
    (axes[0], ts_B, dep_B, mat_OB),
    (axes[1], ts_F, dep_F, mat_OF),
]
last_abs_pc = None
for ax, ts, dep, mat in panel_data_abs:
    pc = ax.pcolormesh(ts, dep, mat.T, cmap=CMAP_ABS,
                       vmin=vmin, vmax=vmax, shading='nearest')
    ax.invert_yaxis()
    ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
    ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
    last_abs_pc = pc

# Difference panel
ax = axes[2]
pc_diff = ax.pcolormesh(common_t, dep_B, diff_mat.T, cmap=CMAP_DIFF,
                        vmin=-dlim, vmax=dlim, shading='nearest')
ax.invert_yaxis()
ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
axes[0].set_ylabel('Depth (m)', fontsize=9)

# Panel letters
axes[0].text(0.02, 0.97, '(a) Baseline', transform=axes[0].transAxes,
             fontsize=9, ha='left', va='top',
             bbox=dict(facecolor='white', edgecolor='none', pad=2.0, alpha=0.8))
axes[1].text(0.02, 0.97, '(b) FPV', transform=axes[1].transAxes,
             fontsize=9, ha='left', va='top',
             bbox=dict(facecolor='white', edgecolor='none', pad=2.0, alpha=0.8))
axes[2].text(0.02, 0.97, '(c) Δ (FPV − Baseline)', transform=axes[2].transAxes,
             fontsize=9, ha='left', va='top',
             bbox=dict(facecolor='white', edgecolor='none', pad=2.0, alpha=0.8))

# Two shared colorbars, mirroring the Chla pattern
fig.subplots_adjust(left=0.06, right=0.86, top=0.95, bottom=0.18, wspace=0.05)
cbar_ax_abs  = fig.add_axes([0.88, 0.18, 0.012, 0.77])
fig.colorbar(last_abs_pc, cax=cbar_ax_abs, label=OXY_LABEL)
cbar_ax_diff = fig.add_axes([0.95, 0.18, 0.012, 0.77])
fig.colorbar(pc_diff, cax=cbar_ax_diff,
             label=f'Δ(FPV − Baseline)')

plt.savefig(os.path.join(RESULTS_DIR, 'DO_profiles_CAB.png'),
            dpi=150, bbox_inches='tight')
plt.show()

<Figure size 900x300 with 5 Axes>

## Section 4.1 - Secchi depth

In [17]:
# ── Secchi depth diagnostic — CAB_averaged, Baseline vs FPV scen02 ──────
# Source: .his (SecchiDept at CAB_averaged). Secchi is a BLOOM diagnostic
# output, not a .map state variable, so this comes from .his only.

CAB_AVG_IDX_B = RB['loc_idx'].get('CAB_averaged')
CAB_AVG_IDX_F = RF['loc_idx'].get('CAB_averaged')

if CAB_AVG_IDX_B is None or CAB_AVG_IDX_F is None:
    print('CAB_averaged monitor not found in .his — cannot plot Secchi diagnostic')
else:
    secchi_si_B = RB['idx'].get('SecchiDept')
    secchi_si_F = RF['idx'].get('SecchiDept')
    if secchi_si_B is None or secchi_si_F is None:
        print('SecchiDept not in .his — cannot plot diagnostic')
    else:
        # Build daily series
        def _secchi_daily(R, loc_idx, si):
            ts = R['data'][:, loc_idx, si].astype(float)
            ts[ts <= 0] = np.nan
            s = pd.Series(ts, index=R['ts_index']).resample('D').mean()
            return s

        secchi_B = _secchi_daily(RB, CAB_AVG_IDX_B, secchi_si_B)
        secchi_F = _secchi_daily(RF, CAB_AVG_IDX_F, secchi_si_F)

        # Trim to window
        secchi_B = secchi_B.loc[SIM_START:SIM_END]
        secchi_F = secchi_F.loc[SIM_START:SIM_END]

        # Monthly means
        sB_monthly = secchi_B.resample('MS').mean()
        sF_monthly = secchi_F.resample('MS').mean()
        common_months = sB_monthly.index.intersection(sF_monthly.index)
        sB_monthly = sB_monthly.loc[common_months]
        sF_monthly = sF_monthly.loc[common_months]
        month_labels = [m.strftime('%b\n%y') for m in common_months]

        # ── Figure: 3 panels (a) daily time series, (b) monthly bars, (c) delta ──
        import matplotlib.gridspec as gridspec
        fig = plt.figure(figsize=(11, 4))
        gs = gridspec.GridSpec(1, 3, figure=fig,
                               width_ratios=[1, 1, 1],
                               wspace=0.15)
        ax_a = fig.add_subplot(gs[0, 0])
        ax_b = fig.add_subplot(gs[0, 1], sharey=ax_a)
        ax_c = fig.add_subplot(gs[0, 2])
        plt.setp(ax_b.get_yticklabels(), visible=False)
        ax_b.set_ylabel('')
        axes = [ax_a, ax_b, ax_c]
        # fig.suptitle('Secchi depth diagnostic — CAB_averaged (.his)\n'
        #              'Baseline vs FPV scenario 02 — daily',
        #              fontsize=10, y=1.00)

        # (a) Daily time series
        ax = axes[0]
        ax.plot(secchi_B.index, secchi_B.values, 'steelblue', lw=1.2, label='Baseline')
        ax.plot(secchi_F.index, secchi_F.values, 'r-', lw=1.2, label='FPV')
        ax.set_ylabel('Secchi depth (m)', fontsize=9)
        ax.set_title('(a) Daily Secchi depth (location 01)',
                     fontsize=9, loc='left')
        ax.invert_yaxis()    # convention: deeper Secchi → cleaner water
        ax.legend(fontsize=8, loc='best')
        ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
        ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
        ax.grid(True, alpha=0.3)

        # (b) Monthly mean bars
        ax = axes[1]
        x = np.arange(len(common_months))
        bar_w = 0.38
        ax.bar(x - bar_w/2, sB_monthly.values, bar_w,
               color='steelblue', edgecolor='black', lw=0.4,
               label='Baseline')
        ax.bar(x + bar_w/2, sF_monthly.values, bar_w,
               color='tomato', edgecolor='black', lw=0.4,
               label='FPV')
        ax.set_xticks(x, labels=month_labels, fontsize=9)
        ax.set_title('(b) Monthly mean Secchi depth (location 01)', fontsize=9, loc='left')
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, axis='y', alpha=0.3)
        ax.tick_params(labelsize=8)

        # (c) Delta (FPV − Baseline)
        ax = axes[2]
        common = secchi_B.index.intersection(secchi_F.index)
        diff = secchi_F.loc[common] - secchi_B.loc[common]
        ax.fill_between(diff.index, diff.values, 0,
                        where=diff.values >= 0, color='blue', alpha=0.5,
                        label='FPV > Baseline (clearer)')
        ax.fill_between(diff.index, diff.values, 0,
                        where=diff.values < 0, color='red', alpha=0.5,
                        label='FPV < Baseline (murkier)')
        ax.axhline(0, color='k', lw=0.7, ls='--', alpha=0.5)
        ax.set_ylabel('Δ Secchi depth (m)\n(FPV − Baseline)', fontsize=9)
        ax.set_title('(c) Δ Secchi depth (FPV − Baseline)', fontsize=9, loc='left')
        ax.legend(fontsize=8, loc='best')
        ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
        ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
        ax.grid(True, alpha=0.3)

        fig.subplots_adjust(left=0.07, right=0.98, top=0.85, bottom=0.18)
        pos_a = ax_a.get_position()
        pos_b = ax_b.get_position()
        gap_ab = 0.01
        ax_b.set_position([pos_a.x0 + pos_a.width + gap_ab,
                           pos_b.y0, pos_b.width, pos_b.height])
        plt.savefig(os.path.join(RESULTS_DIR, 'Secchi_CAB_averaged.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()

        # ── Monthly mean table ──────────────────────────────────────────
        print('\nSecchi depth — monthly means (m), CAB_averaged:')
        print(f'  {"Month":<10} {"Baseline":>10} {"FPV":>10} '
              f'{"Δ":>10} {"% change":>10}')
        print('  ' + '-' * 55)
        for m in common_months:
            b_val = sB_monthly.at[m]; f_val = sF_monthly.at[m]
            d = f_val - b_val
            pct = d / b_val * 100 if b_val > 0 else float('nan')
            print(f'  {m.strftime("%Y-%m"):<10} {b_val:>10.3f} {f_val:>10.3f} '
                  f'{d:>+10.3f} {pct:>+9.1f}%')

        # ── Summary stats ───────────────────────────────────────────────
        print(f'\nSeason summary (Apr–Aug):')
        print(f'  Baseline mean:   {secchi_B.mean():.3f} m')
        print(f'  FPV mean:        {secchi_F.mean():.3f} m')
        print(f'  Δ mean:          {(secchi_F.mean() - secchi_B.mean()):+.3f} m')
        print(f'  Min Δ (FPV most negative): {diff.min():+.3f} m '
              f'on {diff.idxmin().date()}')
        print(f'  Max Δ (FPV most positive): {diff.max():+.3f} m '
              f'on {diff.idxmax().date()}')

<Figure size 1100x400 with 3 Axes>


Secchi depth — monthly means (m), CAB_averaged:
  Month        Baseline        FPV          Δ   % change
  -------------------------------------------------------
  2023-04         4.972      4.843     -0.128      -2.6%
  2023-05         5.360      5.143     -0.217      -4.0%
  2023-06         4.788      3.986     -0.802     -16.8%
  2023-07         4.688      3.827     -0.861     -18.4%
  2023-08         5.496      4.116     -1.379     -25.1%

Season summary (Apr–Aug):
  Baseline mean:   5.063 m
  FPV mean:        4.383 m
  Δ mean:          -0.680 m
  Min Δ (FPV most negative): -1.575 m on 2023-08-07
  Max Δ (FPV most positive): -0.004 m on 2023-04-01


In [47]:
# Create df and export to csv to analyse average values for chapter 5
baseline_secchi = pd.DataFrame(secchi_B, columns = ['baseline'])
baseline_secchi['FPV'] = secchi_F
baseline_secchi.to_csv('baseline_secchi.csv')
baseline_secchi.head(2)

,baseline,FPV
2023-04-01,4.428319,4.424802
2023-04-02,4.476537,4.459422


## Section 5 — Nutrient diagnostics (depth-averaged + profiles)

In [18]:
def plot_nutrient_comparison(ds_B, ds_F, RB, RF, cab_B, cab_F,
                              cab_m, cab_n, save_dir=None):
    # Time series: DO first (panel a), POC1 removed
    nutrients = [
        ('OXY',  'DO (g/m³)\ndepth-avg 0-8m',    'navy'),
        ('NO3',  'NO3 (gN/m³)\ndepth-avg 0-8m',  'steelblue'),
        ('NH4',  'NH4 (gN/m³)\ndepth-avg 0-8m',  'purple'),
        ('PO4',  'PO4 (gP/m³)\ndepth-avg 0-8m',  'red'),
        ('Si',   'Si (gSi/m³)\ndepth-avg 0-8m',  'goldenrod'),
    ]

    # ── Collect time series for CSV export and summary table ────────────
    # Three layer views per variable: surface (K=K_TOP), bottom (K=K_BOT),
    # and depth-averaged (0-8m).
    def surface_series(ds, R, sub, m, n):
        return get_series(ds, R, sub, K_TOP + 1, m=m, n=n)

    def bottom_series(ds, R, sub, m, n):
        return get_series(ds, R, sub, K_BOT + 1, m=m, n=n)

    ts_data = {}
    for sub, _, _ in nutrients:
        ts_data[sub] = {
            'Baseline': {
                'surf': surface_series(ds_B, RB, sub, cab_m, cab_n),
                'bot':  bottom_series(ds_B, RB, sub, cab_m, cab_n),
                'davg': depth_avg(ds_B, RB, sub, cab_B, m=cab_m, n=cab_n),
            },
            'FPV': {
                'surf': surface_series(ds_F, RF, sub, cab_m, cab_n),
                'bot':  bottom_series(ds_F, RF, sub, cab_m, cab_n),
                'davg': depth_avg(ds_F, RF, sub, cab_F, m=cab_m, n=cab_n),
            },
        }

    # Build wide DataFrame with all three layers per variable
    csv_frame = pd.DataFrame()
    for sub, _, _ in nutrients:
        for scen_label in ['Baseline', 'FPV']:
            for layer_key in ['surf', 'bot', 'davg']:
                col = f'{sub}_{layer_key}_{scen_label}'
                csv_frame[col] = ts_data[sub][scen_label][layer_key]
    csv_frame.index.name = 'Date'

    if save_dir:
        csv_path = os.path.join(save_dir, 'Nutrients_timeseries_CAB.csv')
        csv_frame.to_csv(csv_path, float_format='%.5f')
        print(f'Time-series data exported → {csv_path}')

    # ── Time series — 3×2 grid; last slot unused (depth-avg only in plot) ──
    fig, axes = plt.subplots(3, 2, figsize=(9, 7), sharex=True)
    axes_flat = axes.flatten()
    panel_letters = list('abcdef')

    for ax, (sub, label, color), letter in zip(axes_flat, nutrients,
                                                panel_letters):
        vb = ts_data[sub]['Baseline']['davg']
        vf = ts_data[sub]['FPV']['davg']
        ax.plot(vb.index, vb.values, '-',  color=color, lw=1.2,
                alpha=0.7, label='Baseline')
        ax.plot(vf.index, vf.values, '--', color=color, lw=1.2, label='FPV')
        ax.set_ylabel(label, fontsize=8)
        ax.legend(fontsize=8, loc='upper right')
        ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
        ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
        ax.grid(True, alpha=0.3)
        ax.text(0.02, 0.95, f'({letter})', transform=ax.transAxes,
                fontsize=9, ha='left', va='top',
                bbox=dict(facecolor='white', edgecolor='none',
                          pad=2.0, alpha=0.85))

    for j in range(len(nutrients), len(axes_flat)):
        axes_flat[j].set_visible(False)

    # ── Robust fix: re-enable x-tick labels on the bottom of each column ──
    # With sharex=True, matplotlib hides x-tick labels on non-bottom-row
    # panels. With one panel hidden, panel (d) at axes[1, 1] is effectively
    # the bottom of its column. Walk each column from bottom up and re-enable
    # labels on the first visible panel encountered.
    n_rows_grid, n_cols_grid = axes.shape
    for col in range(n_cols_grid):
        for row in range(n_rows_grid - 1, -1, -1):
            if axes[row, col].get_visible():
                ax = axes[row, col]
                plt.setp(ax.get_xticklabels(), visible=True)
                ax.xaxis.set_major_formatter(xfmt)
                ax.xaxis.set_major_locator(xloc)
                ax.tick_params(axis='x', labelsize=8)
                break

    plt.tight_layout()
    if save_dir:
        plt.savefig(os.path.join(save_dir, 'Nutrients_timeseries_CAB.png'),
                    dpi=150, bbox_inches='tight')
    plt.show()

    # ── Seasonal summary — Surface, Bottom, and Depth-averaged ──────────
    SEASONS = {
        'Spring (Apr-May)':  ('2023-04-01', '2023-05-31'),
        'Summer (Jun-Aug)':  ('2023-06-01', '2023-08-31'),
        'Apr-Aug 2023':      ('2023-04-01', '2023-08-31'),
    }

    LAYERS = [('Surface (K_TOP)', 'surf'),
              ('Bottom (K_BOT)',  'bot'),
              ('Depth-avg',        'davg')]

    rows = []
    for sub, _, _ in nutrients:
        for season, (t0, t1) in SEASONS.items():
            for layer_name, layer_key in LAYERS:
                b = ts_data[sub]['Baseline'][layer_key]
                f = ts_data[sub]['FPV'][layer_key]
                b_mean = b.loc[t0:t1].mean()
                f_mean = f.loc[t0:t1].mean()
                diff_abs = f_mean - b_mean
                diff_pct = (diff_abs / b_mean * 100) if b_mean != 0 else np.nan
                rows.append({
                    'Variable':      sub,
                    'Season':        season,
                    'Layer':         layer_name,
                    'Baseline mean': b_mean,
                    'FPV mean':      f_mean,
                    'Δ absolute':    diff_abs,
                    'Δ %':           diff_pct,
                })
    summary = pd.DataFrame(rows)

    print('\n' + '=' * 95)
    print('Seasonal mean — Surface, Bottom, and Depth-averaged at CAB')
    print('=' * 95)
    print(f'  {"Variable":<8} {"Season":<22} {"Layer":<18} '
          f'{"Baseline":>12} {"FPV":>12} {"Δ abs":>12} {"Δ %":>10}')
    print('  ' + '-' * 95)
    for _, r in summary.iterrows():
        print(f'  {r["Variable"]:<8} {r["Season"]:<22} {r["Layer"]:<18} '
              f'{r["Baseline mean"]:>12.4f} {r["FPV mean"]:>12.4f} '
              f'{r["Δ absolute"]:>+12.4f} {r["Δ %"]:>+9.2f}%')

    if save_dir:
        summary_path = os.path.join(save_dir,
                                      'Nutrients_seasonal_summary_CAB.csv')
        summary.to_csv(summary_path, index=False, float_format='%.5f')
        print(f'\nSeasonal summary exported → {summary_path}')

    # ── Depth profiles — formatted to match the DO style ────────────────
    profile_vars = [
        ('NO3',  'Nitrate NO3 (gN/m³)',  'YlGnBu'),
        ('NH4',  'Ammonium NH4 (gN/m³)', 'YlOrRd'),
        ('PO4',  'Phosphate PO4 (gP/m³)', 'viridis'),
        ('Si',   'Silicate Si (gSi/m³)',  'YlOrBr'),
    ]

    for sub, sub_label, cmap in profile_vars:
        ts_B, dep_B, mat_B = get_depth_matrix(ds_B, RB, sub, cab_B,
                                                m=cab_m, n=cab_n)
        ts_F, dep_F, mat_F = get_depth_matrix(ds_F, RF, sub, cab_F,
                                                m=cab_m, n=cab_n)
        if mat_B is None or mat_F is None:
            print(f'Skipping {sub}: no data')
            continue

        common_t = pd.DatetimeIndex(ts_B).intersection(pd.DatetimeIndex(ts_F))
        ib  = [i for i, t in enumerate(ts_B) if t in common_t]
        ifs = [i for i, t in enumerate(ts_F) if t in common_t]
        diff_mat = mat_F[ifs] - mat_B[ib]

        all_abs = np.concatenate([
            mat_B[~np.isnan(mat_B)],
            mat_F[~np.isnan(mat_F)],
        ])
        vmin, vmax = np.nanpercentile(all_abs, [2, 98])
        dlim = np.nanpercentile(np.abs(diff_mat), 95)

        fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharey=True)

        last_abs_pc = None
        for ax, ts, dep, mat in [
            (axes[0], ts_B, dep_B, mat_B),
            (axes[1], ts_F, dep_F, mat_F),
        ]:
            pc = ax.pcolormesh(ts, dep, mat.T, cmap=cmap,
                               vmin=vmin, vmax=vmax, shading='nearest')
            ax.invert_yaxis()
            ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
            ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
            last_abs_pc = pc
        axes[0].set_ylabel('Depth (m)', fontsize=9)

        ax = axes[2]
        pc_diff = ax.pcolormesh(common_t, dep_B, diff_mat.T, cmap='RdBu_r',
                                vmin=-dlim, vmax=dlim, shading='nearest')
        ax.invert_yaxis()
        ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
        ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)

        axes[0].text(0.02, 0.97, '(a) Baseline', transform=axes[0].transAxes,
                     fontsize=9, ha='left', va='top',
                     bbox=dict(facecolor='white', edgecolor='none',
                               pad=2.0, alpha=0.8))
        axes[1].text(0.02, 0.97, '(b) FPV', transform=axes[1].transAxes,
                     fontsize=9, ha='left', va='top',
                     bbox=dict(facecolor='white', edgecolor='none',
                               pad=2.0, alpha=0.8))
        axes[2].text(0.02, 0.97, '(c) Δ (FPV − Baseline)',
                     transform=axes[2].transAxes,
                     fontsize=9, ha='left', va='top',
                     bbox=dict(facecolor='white', edgecolor='none',
                               pad=2.0, alpha=0.8))

        fig.subplots_adjust(left=0.06, right=0.86, top=0.95, bottom=0.18,
                            wspace=0.05)
        cbar_ax_abs  = fig.add_axes([0.87, 0.18, 0.012, 0.77])
        fig.colorbar(last_abs_pc, cax=cbar_ax_abs, label=sub_label)
        cbar_ax_diff = fig.add_axes([0.96, 0.18, 0.012, 0.77])
        fig.colorbar(pc_diff, cax=cbar_ax_diff,
                     label=f'Δ {sub_label}\n(FPV − Baseline)')

        if save_dir:
            plt.savefig(os.path.join(save_dir,
                                       f'Nutrient_profile_{sub}_CAB.png'),
                        dpi=150, bbox_inches='tight')
        plt.show()


plot_nutrient_comparison(ds_B, ds_F, RB, RF, cab_B, cab_F,
                          cab_m, cab_n, save_dir=RESULTS_DIR)

Time-series data exported → <MODEL_DIR>\results_scenarios\Nutrients_timeseries_CAB.csv


<Figure size 900x700 with 6 Axes>


Seasonal mean — Surface, Bottom, and Depth-averaged at CAB
  Variable Season                 Layer                  Baseline          FPV        Δ abs        Δ %
  -----------------------------------------------------------------------------------------------
  OXY      Spring (Apr-May)       Surface (K_TOP)          9.4976       9.8683      +0.3706     +3.90%
  OXY      Spring (Apr-May)       Bottom (K_BOT)           9.5060       9.7466      +0.2406     +2.53%
  OXY      Spring (Apr-May)       Depth-avg                9.4941       9.7712      +0.2771     +2.92%
  OXY      Summer (Jun-Aug)       Surface (K_TOP)          8.5535       9.4176      +0.8640    +10.10%
  OXY      Summer (Jun-Aug)       Bottom (K_BOT)           7.1823       5.5944      -1.5879    -22.11%
  OXY      Summer (Jun-Aug)       Depth-avg                7.5701       6.9735      -0.5967     -7.88%
  OXY      Apr-Aug 2023           Surface (K_TOP)          8.9299       9.5973      +0.6673     +7.47%
  OXY      Apr-Aug

<Figure size 900x300 with 5 Axes>

<Figure size 900x300 with 5 Axes>

<Figure size 900x300 with 5 Axes>

<Figure size 900x300 with 5 Axes>

In [19]:
# ── Consolidated nutrient depth profiles — 4 rows × 3 cols Hovmöller grid ──
# One row per nutrient (NO3, NH4, PO4, Si), columns are Baseline | FPV | Δ.
# Each row has its own colorbar pair (absolute and difference) since
# concentration ranges differ between nutrients.

import matplotlib.gridspec as gridspec

profile_vars = [
    ('NO3',  'Nitrate NO3 (gN/m³)',   'YlGnBu'),
    ('NH4',  'Ammonium NH4 (gN/m³)',  'YlOrRd'),
    ('PO4',  'Phosphate PO4 (gP/m³)', 'viridis'),
    ('Si',   'Silicate Si (gSi/m³)',  'YlOrBr'),
]

n_rows = len(profile_vars)
n_cols = 3   # Baseline, FPV, Δ

fig = plt.figure(figsize=(10, 2.6 * n_rows))
# 5 columns: 3 panels + 2 colorbars (absolute and difference)
gs = gridspec.GridSpec(
    n_rows, n_cols + 2, figure=fig,
    width_ratios=[1, 1, 1, 0.03, 0.03],
    wspace=0.10, hspace=0.10,
)

panel_letters = list('abcdefghijklmnopqrstuvwxyz')
pidx = 0

for row, (sub, sub_label, cmap) in enumerate(profile_vars):
    ts_B, dep_B, mat_B = get_depth_matrix(ds_B, RB, sub, cab_B,
                                            m=cab_m, n=cab_n)
    ts_F, dep_F, mat_F = get_depth_matrix(ds_F, RF, sub, cab_F,
                                            m=cab_m, n=cab_n)
    if mat_B is None or mat_F is None:
        print(f'Skipping {sub}: no data')
        continue

    common_t = pd.DatetimeIndex(ts_B).intersection(pd.DatetimeIndex(ts_F))
    ib  = [i for i, t in enumerate(ts_B) if t in common_t]
    ifs = [i for i, t in enumerate(ts_F) if t in common_t]
    diff_mat = mat_F[ifs] - mat_B[ib]

    all_abs = np.concatenate([
        mat_B[~np.isnan(mat_B)],
        mat_F[~np.isnan(mat_F)],
    ])
    vmin, vmax = np.nanpercentile(all_abs, [2, 98])
    dlim = np.nanpercentile(np.abs(diff_mat), 95)

    # Build the three panels in this row
    ax_B    = fig.add_subplot(gs[row, 0])
    ax_F    = fig.add_subplot(gs[row, 1], sharey=ax_B)
    ax_diff = fig.add_subplot(gs[row, 2], sharey=ax_B)
    plt.setp(ax_F.get_yticklabels(), visible=False)
    plt.setp(ax_diff.get_yticklabels(), visible=False)

    # Baseline panel
    pc_abs = ax_B.pcolormesh(ts_B, dep_B, mat_B.T, cmap=cmap,
                              vmin=vmin, vmax=vmax, shading='nearest')
    ax_B.invert_yaxis()
    ax_B.set_ylabel('Depth (m)', fontsize=9)
    ax_B.set_xlim(xlim)
    ax_B.tick_params(labelsize=7)

    # FPV panel
    ax_F.pcolormesh(ts_F, dep_F, mat_F.T, cmap=cmap,
                    vmin=vmin, vmax=vmax, shading='nearest')
    ax_F.set_xlim(xlim)
    ax_F.tick_params(labelsize=7)

    # Difference panel
    pc_d = ax_diff.pcolormesh(common_t, dep_B, diff_mat.T, cmap='RdBu_r',
                               vmin=-dlim, vmax=dlim, shading='nearest')
    ax_diff.set_xlim(xlim)
    ax_diff.tick_params(labelsize=7)

    # Date formatting only on the bottom row
    if row == n_rows - 1:
        for ax_ in [ax_B, ax_F, ax_diff]:
            ax_.xaxis.set_major_formatter(xfmt)
            ax_.xaxis.set_major_locator(xloc)
    else:
        for ax_ in [ax_B, ax_F, ax_diff]:
            plt.setp(ax_.get_xticklabels(), visible=False)

    # Panel letters
    letter_b = panel_letters[pidx]; pidx += 1
    letter_f = panel_letters[pidx]; pidx += 1
    letter_d = panel_letters[pidx]; pidx += 1

    ax_B.text(0.02, 0.95, f'({letter_b}) Baseline — {sub}',
              transform=ax_B.transAxes, fontsize=8, ha='left', va='top',
              bbox=dict(facecolor='white', edgecolor='none',
                        pad=2.0, alpha=0.85))
    ax_F.text(0.02, 0.95, f'({letter_f}) FPV — {sub}',
              transform=ax_F.transAxes, fontsize=8, ha='left', va='top',
              bbox=dict(facecolor='white', edgecolor='none',
                        pad=2.0, alpha=0.85))
    ax_diff.text(0.02, 0.95, f'({letter_d}) Δ — {sub}',
                  transform=ax_diff.transAxes, fontsize=8, ha='left', va='top',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))

    # Two colorbars per row
    cax_abs  = fig.add_subplot(gs[row, 3])
    fig.colorbar(pc_abs, cax=cax_abs, label=sub_label).ax.tick_params(labelsize=7)
    cax_diff = fig.add_subplot(gs[row, 4])
    fig.colorbar(pc_d, cax=cax_diff,
                 label=f'Δ {sub_label}').ax.tick_params(labelsize=7)

plt.tight_layout()
if RESULTS_DIR:
    plt.savefig(os.path.join(RESULTS_DIR, 'Nutrient_profiles_All_CAB.png'),
                dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1000x1040 with 20 Axes>

## Section 6 — Phytoplankton diagnostics

Species biomass from NetCDF (ecotypes), Chlfa from `.his` (BLOOM-computed,
not in .map), Secchi from `.his` SecchiDept.

In [43]:
SPECIES_ECOTYPES = {
    'Diatoms':       ['FDIATOMS_E', 'FDIATOMS_N', 'FDIATOMS_P'],
    'Flagellates':   ['FFLAGELA'],
    'Greens':        ['GREENS_E', 'GREENS_N', 'GREENS_P'],
    'Cyanobacteria': ['MICROCYS_E', 'MICROCYS_N', 'MICROCYS_P',
                      'APHANFIX_E', 'APHANFIX_N', 'APHANFIX_P', 'APHANFIX_F',
                      'OSCILAT_E', 'OSCILAT_N', 'OSCILAT_P',
                      'ANABAENA_E', 'ANABAENA_N', 'ANABAENA_P'],
}
ALL_ALG = [s for v in SPECIES_ECOTYPES.values() for s in v]

GROUPS_ORDER  = ['Diatoms', 'Flagellates', 'Greens', 'Cyanobacteria']
GROUP_COLOURS = ['#ff7f0e', '#1f77b4', '#2ca02c', '#9467bd']


def species_sum_series(ds, his_run, ecotypes, k_label, m, n, loc_dict):
    """Sum all available ecotypes — daily-resampled."""
    cols = []
    for sp in ecotypes:
        s = get_series(ds, his_run, sp, k_label, m=m, n=n, loc_dict=loc_dict)
        if not s.empty:
            cols.append(s)
    if not cols:
        return pd.Series(dtype=float)
    df = pd.concat(cols, axis=1).fillna(0)
    return df.sum(axis=1)


def build_group_df(ds, his_run, cab_dict, m, n, k_label=8):
    """DataFrame indexed by date, columns = species groups."""
    cols = {}
    for grp in GROUPS_ORDER:
        cols[grp] = species_sum_series(ds, his_run, SPECIES_ECOTYPES[grp],
                                        k_label, m, n, cab_dict)
    return pd.concat(cols, axis=1).fillna(0)


def build_greens_ecotype_df(ds, his_run, cab_dict, m, n, k_label=8):
    """GREENS_E, GREENS_N, GREENS_P daily — used for P-lim shading."""
    cols = {}
    for sp in ['GREENS_E', 'GREENS_N', 'GREENS_P']:
        cols[sp] = get_series(ds, his_run, sp, k_label,
                              m=m, n=n, loc_dict=cab_dict)
    return pd.concat(cols, axis=1).fillna(0)


def identify_plim_period(greens_df, threshold=0.20):
    """Boolean: True where GREENS_P / total_GREENS > threshold."""
    total = greens_df.sum(axis=1).replace(0, np.nan)
    frac_p = greens_df['GREENS_P'] / total
    return (frac_p > threshold).fillna(False)


# ── Build daily group series for both scenarios ─────────────────────────
groups_B = build_group_df(ds_B, RB, cab_B, cab_m, cab_n, k_label=8)
groups_F = build_group_df(ds_F, RF, cab_F, cab_m, cab_n, k_label=8)
greens_B = build_greens_ecotype_df(ds_B, RB, cab_B, cab_m, cab_n, k_label=8)
greens_F = build_greens_ecotype_df(ds_F, RF, cab_F, cab_m, cab_n, k_label=8)

plim_B = identify_plim_period(greens_B, threshold=0.20)
plim_F = identify_plim_period(greens_F, threshold=0.20)

# Monthly means
groups_B_M = groups_B.resample('MS').mean()
groups_F_M = groups_F.resample('MS').mean()
common_months = groups_B_M.index.intersection(groups_F_M.index)
groups_B_M = groups_B_M.loc[common_months]
groups_F_M = groups_F_M.loc[common_months]
month_labels = [m.strftime('%b\n%y') for m in common_months]


# ── Figure: 3 rows × 2 cols (panel (f) removed) ─────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(8, 8))
# fig.suptitle('Species composition analysis — Baseline vs FPV scenario 02\n'
#              'CAB layer 8 (0.16 m) — daily mean',
#              fontsize=11, y=0.995)

def shade_plim(ax, plim_series, color='red', alpha=0.10):
    """Shade contiguous True ranges of plim_series."""
    if plim_series.empty:
        return
    plim_int = plim_series.astype(int).values
    idx = plim_series.index
    in_run = False; start = None; first_label_used = False
    for i, v in enumerate(plim_int):
        if v and not in_run:
            start = idx[i]; in_run = True
        elif not v and in_run:
            ax.axvspan(start, idx[i-1], color=color, alpha=alpha,
                       label=None if first_label_used else 'P-lim period')
            first_label_used = True
            in_run = False
    if in_run:
        ax.axvspan(start, idx[-1], color=color, alpha=alpha,
                   label=None if first_label_used else 'P-lim period')


# ── (a) Baseline stacked biomass ────────────────────────────────────────
ax = axes[0, 0]
stack_B = [groups_B[grp].values for grp in GROUPS_ORDER]
ax.stackplot(groups_B.index, stack_B, labels=GROUPS_ORDER,
             colors=GROUP_COLOURS, alpha=0.85)
shade_plim(ax, plim_B, color='red', alpha=0.10)
ax.set_ylabel('Biomass (gC/m³)', fontsize=9)
ax.set_title('(a) Baseline — total biomass',
             fontsize=9, loc='left')
ax.legend(fontsize=7, loc='upper left')
ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
ax.grid(True, alpha=0.3)

# ── (b) FPV stacked biomass ─────────────────────────────────────────────
ax = axes[0, 1]
stack_F = [groups_F[grp].values for grp in GROUPS_ORDER]
ax.stackplot(groups_F.index, stack_F, labels=GROUPS_ORDER,
             colors=GROUP_COLOURS, alpha=0.85)
shade_plim(ax, plim_F, color='red', alpha=0.10)
ax.set_ylabel('Biomass (gC/m³)', fontsize=9)
ax.set_title('(b) FPV scen02 — total biomass',
             fontsize=9, loc='left')
ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
ax.grid(True, alpha=0.3)


# ── (c) Monthly mean biomass — grouped + stacked bars ────────────────────
ax = axes[1, 0]
x = np.arange(len(common_months))
bar_w = 0.38
bottom_B = np.zeros(len(common_months))
bottom_F = np.zeros(len(common_months))
for grp, col in zip(GROUPS_ORDER, GROUP_COLOURS):
    vals_B = groups_B_M[grp].values
    vals_F = groups_F_M[grp].values
    ax.bar(x - bar_w/2, vals_B, bar_w, bottom=bottom_B,
           color=col, edgecolor='black', lw=0.6, alpha=0.95, label=grp)
    ax.bar(x + bar_w/2, vals_F, bar_w, bottom=bottom_F,
           color=col, edgecolor='black', lw=0.6, alpha=0.45, hatch='//')
    bottom_B = bottom_B + vals_B
    bottom_F = bottom_F + vals_F

style_handles = [
    Patch(facecolor='grey', edgecolor='black', lw=0.6, alpha=0.95,
          label='Baseline (solid)'),
    Patch(facecolor='grey', edgecolor='black', lw=0.6, alpha=0.45,
          hatch='//', label='FPV (hatched)'),
]
grp_handles = [Patch(facecolor=col, edgecolor='black', lw=0.4, label=grp)
               for grp, col in zip(GROUPS_ORDER, GROUP_COLOURS)]
ax.legend(handles=grp_handles + style_handles,
          fontsize=7, loc='upper left', ncol=1)
ax.set_xticks(x); ax.set_xticklabels(month_labels)
ax.set_ylabel('Mean biomass (gC/m³)', fontsize=9)
ax.set_title('(c) Monthly mean biomass — Baseline vs FPV',
             fontsize=9, loc='left')
ax.tick_params(labelsize=8)
ax.grid(True, axis='y', alpha=0.3)


# ── (d) Monthly % composition — grouped + stacked bars ──────────────────
ax = axes[1, 1]
total_B = groups_B_M.sum(axis=1).replace(0, np.nan).values
total_F = groups_F_M.sum(axis=1).replace(0, np.nan).values
bottom_B = np.zeros(len(common_months))
bottom_F = np.zeros(len(common_months))
for grp, col in zip(GROUPS_ORDER, GROUP_COLOURS):
    vals_B = groups_B_M[grp].values / total_B * 100
    vals_F = groups_F_M[grp].values / total_F * 100
    ax.bar(x - bar_w/2, vals_B, bar_w, bottom=bottom_B,
           color=col, edgecolor='black', lw=0.6, alpha=0.95)
    ax.bar(x + bar_w/2, vals_F, bar_w, bottom=bottom_F,
           color=col, edgecolor='black', lw=0.6, alpha=0.45, hatch='//')
    bottom_B = bottom_B + np.nan_to_num(vals_B)
    bottom_F = bottom_F + np.nan_to_num(vals_F)
ax.set_xticks(x); ax.set_xticklabels(month_labels)
ax.set_ylim(0, 105)
ax.set_ylabel('% of total phytoplankton biomass', fontsize=9)
ax.set_title('(d) Monthly group composition',
             fontsize=9, loc='left')
ax.tick_params(labelsize=8)
ax.grid(True, axis='y', alpha=0.3)


# ── (e) Biomass change (FPV − Baseline), all groups ─────────────────────
ax = axes[2, 0]
common_idx = groups_B.index.intersection(groups_F.index)
for grp, col in zip(GROUPS_ORDER, GROUP_COLOURS):
    diff = groups_F[grp].loc[common_idx] - groups_B[grp].loc[common_idx]
    ax.plot(diff.index, diff.values, color=col, lw=1.1, label=grp)
ax.axhline(0, color='k', lw=0.7, ls='--', alpha=0.5)
ax.set_ylabel('Biomass change (gC/m³)\nΔ(FPV − Baseline)', fontsize=9)
ax.set_title('(e) Biomass change (FPV − Baseline)',
             fontsize=9, loc='left')
ax.legend(fontsize=7, loc='upper left', ncol=1)
ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
ax.grid(True, alpha=0.3)


# ── (f) — slot empty (panel removed) ────────────────────────────────────
axes[2, 1].set_visible(False)

# ── Footer ──────────────────────────────────────────────────────────────
# fig.text(0.5, 0.005,
#          'Total carbon biomass — all BLOOM ecotypes summed (gC/m³).  '
#          'Red shading (a)(b) = P-limitation period (P-type share > 20%).  '
#          'Bars: solid = Baseline, faded/dashed = FPV.',
#          ha='center', fontsize=7, color='grey')
# ── Share y across panels (a) (b) (c) (e) ───────────────────────────────
shared_axes = [axes[0, 0], axes[0, 1], axes[1, 0], axes[2, 0]]
# Compute pooled y-range across all four panels
y_min = min(ax.get_ylim()[0] for ax in shared_axes)
y_max = max(ax.get_ylim()[1] for ax in shared_axes)
for ax in shared_axes:
    ax.set_ylim(y_min, y_max)
    
plt.tight_layout(rect=[0, 0.02, 1, 0.985])
plt.savefig(os.path.join(RESULTS_DIR, 'Phyto_timeseries_CAB.png'),
            dpi=150, bbox_inches='tight')
plt.show()

<Figure size 800x800 with 6 Axes>

In [21]:
# Option A — Biomass-weighted limitation across ALL algal groups (E/N/P/F)
# Sum E-, N-, P-, and (where present) F-adapted ecotypes across Diatoms,
# Greens, and Cyanobacteria. Flagellates have no E/N/P split — excluded.

# Note: APHANFIX has an F (nitrogen-fixing, free?) ecotype that doesn't
# fit E/N/P. Tracked separately and reported if non-zero.

import matplotlib.gridspec as gridspec

ALL_E = ['FDIATOMS_E', 'GREENS_E',
         'MICROCYS_E', 'APHANFIX_E', 'OSCILAT_E', 'ANABAENA_E']
ALL_N = ['FDIATOMS_N', 'GREENS_N',
         'MICROCYS_N', 'APHANFIX_N', 'OSCILAT_N', 'ANABAENA_N']
ALL_P = ['FDIATOMS_P', 'GREENS_P',
         'MICROCYS_P', 'APHANFIX_P', 'OSCILAT_P', 'ANABAENA_P']
ALL_F = ['APHANFIX_F']   # nitrogen-fixing free ecotype

ECOTYPE_COLOURS = {
    'E': '#2ca02c',   # green — energy/light-adapted
    'N': '#1f77b4',   # blue  — N-adapted
    'P': '#d62728',   # red   — P-adapted
    'F': '#9467bd',   # purple — N-fixing (free)
}
ECOTYPE_LABELS = {
    'E': 'E (light-adapted)',
    'N': 'N (N-adapted)',
    'P': 'P (P-adapted)',
    'F': 'F (N-fixing free)',
}

def build_ecotype_df(ds, his_run, cab_dict, m, n, k_label=8):
    """DataFrame with columns E, N, P, F — total carbon biomass by ecotype
    summed across all available algal groups. Indexed by datetime even when
    some columns are empty."""
    cols = {}
    cols['E'] = species_sum_series(ds, his_run, ALL_E, k_label, m, n, cab_dict)
    cols['N'] = species_sum_series(ds, his_run, ALL_N, k_label, m, n, cab_dict)
    cols['P'] = species_sum_series(ds, his_run, ALL_P, k_label, m, n, cab_dict)
    cols['F'] = species_sum_series(ds, his_run, ALL_F, k_label, m, n, cab_dict)
    # Find a reference DatetimeIndex from any non-empty column
    ref_index = None
    for s in cols.values():
        if not s.empty:
            ref_index = s.index
            break
    if ref_index is None:
        return pd.DataFrame()
    # Reindex every column (empty ones get filled with zeros on the ref index)
    for k, s in cols.items():
        if s.empty:
            cols[k] = pd.Series(0.0, index=ref_index)
    df = pd.concat(cols, axis=1).fillna(0)
    df.index = pd.DatetimeIndex(df.index)
    return df

ecot_B = build_ecotype_df(ds_B, RB, cab_B, cab_m, cab_n, k_label=8)
ecot_F = build_ecotype_df(ds_F, RF, cab_F, cab_m, cab_n, k_label=8)

# Drop F column from plot if it's effectively zero everywhere
f_max = max(ecot_B['F'].max(), ecot_F['F'].max())
plot_ecotypes = ['E', 'N', 'P'] + (['F'] if f_max > 1e-6 else [])
print(f'F-ecotype max biomass: {f_max:.6g} gC/m³  '
      f'({"included" if f_max > 1e-6 else "skipped — effectively zero"})')


# ── Figure setup with gridspec ──────────────────────────────────────────
fig = plt.figure(figsize=(11, 3.5))
gs = gridspec.GridSpec(1, 3, figure=fig,
                       width_ratios=[1, 1, 1],
                       wspace=0.15)             # default gap for (b)↔(c)

ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1], sharey=ax_a)  # share y with (a)
ax_c = fig.add_subplot(gs[0, 2])               # independent y

# Hide y-tick labels on (b), keep ticks
plt.setp(ax_b.get_yticklabels(), visible=False)
ax_b.set_ylabel('')                            # remove redundant "Biomass" label on (b)

axes = [ax_a, ax_b, ax_c]                      # keep existing references working


# (a) Baseline
ax = axes[0]
stack = [ecot_B[e].values for e in plot_ecotypes]
colours = [ECOTYPE_COLOURS[e] for e in plot_ecotypes]
labels = [ECOTYPE_LABELS[e] for e in plot_ecotypes]
ax.stackplot(ecot_B.index, stack, labels=labels, colors=colours, alpha=0.85)
ax.set_ylabel('Biomass (gC/m³)', fontsize=9)
ax.set_title('(a) Baseline — ecotype carbon biomass', fontsize=9, loc='left')
ax.legend(fontsize=7, loc='upper left')
ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
ax.grid(True, alpha=0.3)

# (b) FPV
ax = axes[1]
stack = [ecot_F[e].values for e in plot_ecotypes]
ax.stackplot(ecot_F.index, stack, labels=labels, colors=colours, alpha=0.85)
ax.set_title('(b) FPV — ecotype carbon biomass', fontsize=9, loc='left')
ax.legend(fontsize=7, loc='upper left')
ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
ax.grid(True, alpha=0.3)

# (c) Delta = FPV − Baseline (per ecotype, line plot)
ax = axes[2]
common = ecot_B.index.intersection(ecot_F.index)
for e in plot_ecotypes:
    diff = ecot_F[e].loc[common] - ecot_B[e].loc[common]
    ax.plot(diff.index, diff.values, color=ECOTYPE_COLOURS[e], lw=1.2,
            label=ECOTYPE_LABELS[e])
ax.axhline(0, color='k', lw=0.7, ls='--', alpha=0.5)
ax.set_ylabel('ΔFPV − Baseline (gC/m³)', fontsize=9)
ax.set_title('(c) Δ ecotype biomass (FPV − Baseline)', fontsize=9, loc='left')
ax.legend(fontsize=7, loc='upper left')
ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
ax.grid(True, alpha=0.3)


# ── Manually tighten only the (a)-(b) pair ──────────────────────────────
# Apply this AFTER all plotting so positions are stable
fig.subplots_adjust(left=0.07, right=0.98, top=0.90, bottom=0.22)
pos_a = ax_a.get_position()
pos_b = ax_b.get_position()
gap_ab = 0.01
ax_b.set_position([pos_a.x0 + pos_a.width + gap_ab,
                   pos_b.y0, pos_b.width, pos_b.height])

fig.text(0.5, 0.02,
         'Interpretation: dominance of an ecotype indicates which limitation '
         'regime the standing biomass evolved under.\nP-adapted dominance → P-limited population.',
         ha='center', fontsize=8, color='grey')

plt.savefig(os.path.join(RESULTS_DIR, 'Limitation_OptionA_biomass_ecotypes.png'),
            dpi=150, bbox_inches='tight')
plt.show()


# Monthly means
print('\nEcotype biomass — monthly means (gC/m³):')
print(f'  {"Scenario":<10} {"Month":<10}', end='')
for e in plot_ecotypes:
    print(f'  {ECOTYPE_LABELS[e][:5]:>8}', end='')
print()
print('  ' + '-' * 80)
for scen_name, df in [('Baseline', ecot_B), ('FPV', ecot_F)]:
    monthly = df.resample('MS').mean()
    for month, row in monthly.iterrows():
        print(f'  {scen_name:<10} {month.strftime("%Y-%m"):<10}', end='')
        for e in plot_ecotypes:
            print(f'  {row[e]:>8.4f}', end='')
        print()
    print()

F-ecotype max biomass: 0 gC/m³  (skipped — effectively zero)


<Figure size 1100x350 with 3 Axes>


Ecotype biomass — monthly means (gC/m³):
  Scenario   Month          E (li     N (N-     P (P-
  --------------------------------------------------------------------------------
  Baseline   2023-04       0.0721    0.0002    0.0040
  Baseline   2023-05       0.0823    0.0000    0.0000
  Baseline   2023-06       0.1829    0.0000    0.4727
  Baseline   2023-07       0.2062    0.0000    0.2951
  Baseline   2023-08       0.1033    0.0000    0.0000

  FPV        2023-04       0.0855    0.0002    0.0041
  FPV        2023-05       0.1513    0.0002    0.0252
  FPV        2023-06       0.0210    0.0001    1.1512
  FPV        2023-07       0.0009    0.0000    1.0144
  FPV        2023-08       0.0983    0.0001    0.7057



In [22]:
# Option B — BLOOM's internal limitation diagnostics (.his, CAB layer 8)
# Filtered to the E/N/P scheme that matches Option A:
#   Limit nit ↔ N adaptation,  Limit pho ↔ P adaptation,  Limit e ↔ E adaptation
# Values are 0-1; the MINIMUM across these three is the active growth limit.

import matplotlib.gridspec as gridspec

LIMIT_VARS = {
    'Limit nit':  ('N limitation',      '#1f77b4'),   # blue
    'Limit pho':  ('P limitation',      '#d62728'),   # red
    'Limit e':    ('Light lim.', '#2ca02c'),   # green
}


def his_limit_series(his_run, cab_dict, var_name, k_label=8):
    """Read a Limit* series at CAB(k_label) from .his. Daily resampled."""
    si = his_run['idx'].get(var_name)
    if si is None or k_label not in cab_dict:
        return pd.Series(dtype=float)
    ts = pd.Series(his_run['data'][:, cab_dict[k_label], si].astype(float),
                   index=his_run['ts_index'])
    return ts.resample('D').mean()


# Build daily Limit* series for both scenarios
limit_B = pd.DataFrame({
    v: his_limit_series(RB, cab_B, v, k_label=8) for v in LIMIT_VARS
})
limit_F = pd.DataFrame({
    v: his_limit_series(RF, cab_F, v, k_label=8) for v in LIMIT_VARS
})

# Drop any that are entirely NaN
limit_B = limit_B.dropna(axis=1, how='all')
limit_F = limit_F.dropna(axis=1, how='all')
present = [v for v in LIMIT_VARS if v in limit_B.columns and v in limit_F.columns]
print(f'Limit* variables present in .his: {present}')


# ── Figure setup with gridspec ──────────────────────────────────────────
fig = plt.figure(figsize=(11, 3.5))
gs = gridspec.GridSpec(1, 3, figure=fig,
                       width_ratios=[1, 1, 1],
                       wspace=0.12)             # default gap for (b)↔(c)

ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1], sharey=ax_a)  # share y with (a)
ax_c = fig.add_subplot(gs[0, 2])               # independent y

# Hide y-tick labels on (b), keep ticks
plt.setp(ax_b.get_yticklabels(), visible=False)
ax_b.set_ylabel('')                            # remove redundant y-label on (b)

axes = [ax_a, ax_b, ax_c]


# (a) Baseline
ax = axes[0]
for v in present:
    label, colour = LIMIT_VARS[v]
    ax.plot(limit_B.index, limit_B[v].values, color=colour, lw=1.1, label=label)
ax.axhline(1, color='k', lw=0.5, ls=':', alpha=0.5)
ax.set_ylabel('Limitation factor (0-1)', fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_title('(a) Baseline — Limit* factors (E/N/P only)', fontsize=9, loc='left')
ax.legend(fontsize=7, loc='upper left')
ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
ax.grid(True, alpha=0.3)

# (b) FPV
ax = axes[1]
for v in present:
    label, colour = LIMIT_VARS[v]
    ax.plot(limit_F.index, limit_F[v].values, color=colour, lw=1.1, label=label)
ax.axhline(1, color='k', lw=0.5, ls=':', alpha=0.5)
ax.set_title('(b) FPV — Limit* factors (E/N/P only)', fontsize=9, loc='left')
ax.legend(fontsize=7, loc='upper left')
ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
ax.grid(True, alpha=0.3)

# (c) Delta (FPV − Baseline)
ax = axes[2]
common = limit_B.index.intersection(limit_F.index)
for v in present:
    label, colour = LIMIT_VARS[v]
    diff = limit_F[v].loc[common] - limit_B[v].loc[common]
    ax.plot(diff.index, diff.values, color=colour, lw=1.1, label=label)
ax.axhline(0, color='k', lw=0.7, ls='--', alpha=0.5)
ax.set_ylabel('ΔFPV − Baseline', fontsize=9)
ax.set_title('(c) Δ Limit* factors (FPV − Baseline)', fontsize=9, loc='left')
ax.legend(fontsize=7, loc='upper left')
ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
ax.grid(True, alpha=0.3)


# ── Manually tighten only the (a)-(b) pair ──────────────────────────────
fig.subplots_adjust(left=0.07, right=0.98, top=0.88, bottom=0.22)
pos_a = ax_a.get_position()
pos_b = ax_b.get_position()
gap_ab = 0.01
ax_b.set_position([pos_a.x0 + pos_a.width + gap_ab,
                   pos_b.y0, pos_b.width, pos_b.height])

# Suptitle (placed via fig.text so it doesn't clip with subplots_adjust)
# fig.text(0.5, 0.96,
#          'Option B — BLOOM Limit* diagnostics (.his, CAB layer 8)\n'
#          'E/N/P scheme matching Option A — minimum across these drives growth',
#          ha='center', fontsize=10)

fig.text(0.5, 0.02,
         'Interpretation: low Limit* values mean strong limitation. The minimum '
         'across N/P/E is the active rate-limiter at each timestep.\n'
         'Delta: positive Δ = FPV relaxes that limit.',
         ha='center', fontsize=8, color='grey')

plt.savefig(os.path.join(RESULTS_DIR, 'Limitation_OptionB_BLOOM_diagnostic.png'),
            dpi=150, bbox_inches='tight')
plt.show()


# Active limiter summary across N/P/E only
print('\nActive growth limiter (most restrictive across N/P/E):')
for scen_name, df in [('Baseline', limit_B), ('FPV', limit_F)]:
    active_lim = df[present].idxmin(axis=1)
    share = active_lim.value_counts(normalize=True) * 100
    print(f'  {scen_name}:')
    for v, pct in share.items():
        label, _ = LIMIT_VARS[v]
        print(f'    {label:<22} {pct:>5.1f}% of timesteps')

# Monthly mean of each limit factor
print('\nMonthly mean limitation factors:')
print(f'  {"Scenario":<10} {"Month":<10}', end='')
for v in present:
    label, _ = LIMIT_VARS[v]
    print(f'  {label[:6]:>8}', end='')
print()
print('  ' + '-' * 70)
for scen_name, df in [('Baseline', limit_B), ('FPV', limit_F)]:
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    monthly = df.resample('MS').mean()
    for month, row in monthly.iterrows():
        print(f'  {scen_name:<10} {month.strftime("%Y-%m"):<10}', end='')
        for v in present:
            print(f'  {row[v]:>8.4f}', end='')
        print()
    print()

Limit* variables present in .his: ['Limit nit', 'Limit pho', 'Limit e']


<Figure size 1100x350 with 3 Axes>


Active growth limiter (most restrictive across N/P/E):
  Baseline:
    N limitation           100.0% of timesteps
  FPV:
    N limitation           100.0% of timesteps

Monthly mean limitation factors:
  Scenario   Month         N limi    P limi    Light 
  ----------------------------------------------------------------------
  Baseline   2023-04       0.0000    0.0000    0.0000
  Baseline   2023-05       0.0000    0.0000    0.0000
  Baseline   2023-06       0.0000    0.6667    0.0000
  Baseline   2023-07       0.0000    0.6452    0.0000
  Baseline   2023-08       0.0000    0.0000    0.0000

  FPV        2023-04       0.0000    0.0000    0.0000
  FPV        2023-05       0.0000    0.0645    0.0000
  FPV        2023-06       0.0000    1.0000    0.0000
  FPV        2023-07       0.0000    1.0000    0.0000
  FPV        2023-08       0.0000    0.8710    0.0000



In [23]:
# ── Chlorophyll a — Hovmöller depth profiles at CAB and BIS ──────────────
# Reads Chlfa from .his at each CAB(N) and BIS(N) layer monitor and stacks
# into a (time, depth) matrix. Same approach as Section 5's DO profiles.

def chlfa_depth_matrix(his_run, loc_dict):
    """Build (n_times, n_layers) Chlfa matrix at CAB or BIS across active
    layers. Self-contained: reads Chlfa from .his directly, with fallback
    to ecotype reconstruction."""
    layers = [k for k in sorted(loc_dict.keys())
              if his_k_to_waq(k) in range(K_TOP, K_BOT + 1)]
    if not layers:
        return None, None, None
    depths = [LAYER_MID_DEPTHS[his_k_to_waq(k)] for k in layers]

    cols = []
    for k_label in layers:
        # Try Chlfa directly from .his
        si = his_run['idx'].get('Chlfa')
        if si is not None and k_label in loc_dict:
            ts = pd.Series(
                his_run['data'][:, loc_dict[k_label], si].astype(float),
                index=his_run['ts_index']).resample('D').mean()
        else:
            # Fallback: sum all algae ecotypes × ChlaCAlg
            algae_subs = [s for s in his_run['substances']
                          if s.endswith(('_E', '_N', '_P', '_F'))
                          or s == 'FFLAGELA']
            ts = None
            for sp in algae_subs:
                sp_si = his_run['idx'].get(sp)
                if sp_si is None or k_label not in loc_dict:
                    continue
                contrib = pd.Series(
                    his_run['data'][:, loc_dict[k_label], sp_si].astype(float),
                    index=his_run['ts_index']
                ).resample('D').mean() * ChlaCAlg * 1000
                ts = contrib if ts is None else ts.add(contrib, fill_value=0)
            if ts is None:
                ts = pd.Series(dtype=float)
        cols.append(ts)

    if not cols or all(c.empty for c in cols):
        return None, None, None
    df = pd.concat(cols, axis=1)
    df.columns = layers
    return df.index, depths, df.values


# Build matrices for both stations
ts_CAB_B, dep_CAB_B, chla_CAB_B = chlfa_depth_matrix(RB, cab_B)
ts_CAB_F, dep_CAB_F, chla_CAB_F = chlfa_depth_matrix(RF, cab_F)
ts_BIS_B, dep_BIS_B, chla_BIS_B = chlfa_depth_matrix(RB, bis_B)
ts_BIS_F, dep_BIS_F, chla_BIS_F = chlfa_depth_matrix(RF, bis_F)

if chla_CAB_B is None or chla_BIS_B is None:
    print('Could not build Chlfa depth matrices — check that CAB/BIS layer '
          'monitors are present in .his and Chlfa is in the substance list.')
else:
    # Shared colour scale across CAB and BIS (absolute), and across deltas
    all_abs = np.concatenate([
        chla_CAB_B[~np.isnan(chla_CAB_B)],
        chla_CAB_F[~np.isnan(chla_CAB_F)],
        chla_BIS_B[~np.isnan(chla_BIS_B)],
        chla_BIS_F[~np.isnan(chla_BIS_F)],
    ])
    vmin, vmax = np.nanpercentile(all_abs, [2, 98])

    # Difference matrices on common times
    def _make_diff(ts_B, mat_B, ts_F, mat_F):
        common = pd.DatetimeIndex(ts_B).intersection(pd.DatetimeIndex(ts_F))
        ib = [i for i, t in enumerate(ts_B) if t in common]
        i_f = [i for i, t in enumerate(ts_F) if t in common]
        return common, mat_F[i_f] - mat_B[ib]

    common_CAB, diff_CAB = _make_diff(ts_CAB_B, chla_CAB_B,
                                      ts_CAB_F, chla_CAB_F)
    common_BIS, diff_BIS = _make_diff(ts_BIS_B, chla_BIS_B,
                                      ts_BIS_F, chla_BIS_F)
    all_diff = np.concatenate([diff_CAB[~np.isnan(diff_CAB)],
                                diff_BIS[~np.isnan(diff_BIS)]])
    dlim = np.nanpercentile(np.abs(all_diff), 98) if all_diff.size else 1.0

    # ── 2 rows (CAB / BIS) × 3 cols (Baseline / FPV / Δ) ──────────────────
    fig, axes = plt.subplots(2, 3, figsize=(10, 6), sharey=True)

    panel_data = [
        # (row, col, ax, ts, dep, mat, station, title_label, is_diff)
        (0, 0, axes[0, 0], ts_CAB_B, dep_CAB_B, chla_CAB_B, 'CAB',
         '(a) Baseline — Loc 01', False),
        (0, 1, axes[0, 1], ts_CAB_F, dep_CAB_F, chla_CAB_F, 'CAB',
         '(b) FPV — Loc 01', False),
        (0, 2, axes[0, 2], common_CAB, dep_CAB_B, diff_CAB, 'CAB',
         '(c) Δ (FPV − Baseline) — Loc 01', True),
        (1, 0, axes[1, 0], ts_BIS_B, dep_BIS_B, chla_BIS_B, 'BIS',
         '(d) Baseline — Loc 02', False),
        (1, 1, axes[1, 1], ts_BIS_F, dep_BIS_F, chla_BIS_F, 'BIS',
         '(e) FPV — Loc 02', False),
        (1, 2, axes[1, 2], common_BIS, dep_BIS_B, diff_BIS, 'BIS',
         '(f) Δ (FPV − Baseline) — Loc 02', True),
    ]

    last_abs_pc = None
    last_diff_pc = None
    for row, col, ax, ts, dep, mat, station, title, is_diff in panel_data:
        cmap = 'RdBu_r' if is_diff else 'YlGn'
        v0 = -dlim if is_diff else vmin
        v1 = dlim if is_diff else vmax
        pc = ax.pcolormesh(ts, dep, mat.T, cmap=cmap,
                           vmin=v0, vmax=v1, shading='nearest')
        ax.set_title(title, fontsize=9, loc='left')
        if col == 0:
            ax.set_ylabel('Depth (m)', fontsize=9)
        ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
        ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=8)
        if is_diff:
            last_diff_pc = pc
        else:
            last_abs_pc = pc
    ax.invert_yaxis()
    # Two colorbars: one for absolute (cols 0-1) one for delta (col 2)
    fig.subplots_adjust(left=0.06, right=0.86, top=0.93, bottom=0.10,
                        hspace=0.30, wspace=0.15)
    cbar_ax_abs  = fig.add_axes([0.88, 0.10, 0.012, 0.83])
    cbar_ax_diff = fig.add_axes([0.95, 0.10, 0.012, 0.83])

    fig.colorbar(last_abs_pc, cax=cbar_ax_abs,
                 label=label_for('Chlfa') if 'label_for' in dir()
                       else 'Chl a (µg/L)')
    fig.colorbar(last_diff_pc, cax=cbar_ax_diff,
                 label=f'Δ Chl a (µg/L)\n(FPV − Baseline)')

    plt.savefig(os.path.join(RESULTS_DIR, 'Chla_depth_profiles_CAB_BIS.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    # ── Summary stats ──────────────────────────────────────────────────────
    print('\nChla depth profile summary (Apr–Aug 2023):')
    print(f'  {"Station":<8} {"Scenario":<10} {"Surface mean":>14} '
          f'{"Bottom mean":>14} {"Column mean":>14}')
    print('  ' + '-' * 65)
    for station, ts, dep, mat in [
        ('CAB', ts_CAB_B, dep_CAB_B, chla_CAB_B),
        ('CAB', ts_CAB_F, dep_CAB_F, chla_CAB_F),
        ('BIS', ts_BIS_B, dep_BIS_B, chla_BIS_B),
        ('BIS', ts_BIS_F, dep_BIS_F, chla_BIS_F),
    ]:
        scen = 'Baseline' if mat is chla_CAB_B or mat is chla_BIS_B else 'FPV'
        surf_mean = np.nanmean(mat[:, 0])
        bot_mean  = np.nanmean(mat[:, -1])
        col_mean  = np.nanmean(mat)
        print(f'  {station:<8} {scen:<10} {surf_mean:>14.3f} '
              f'{bot_mean:>14.3f} {col_mean:>14.3f}')

<Figure size 1000x600 with 8 Axes>


Chla depth profile summary (Apr–Aug 2023):
  Station  Scenario     Surface mean    Bottom mean    Column mean
  -----------------------------------------------------------------
  CAB      Baseline            8.779          0.406          2.678
  CAB      FPV                17.317          0.663          5.516
  BIS      Baseline            8.751          0.410          2.669
  BIS      FPV                17.214          0.670          5.484


In [39]:
# ── Chlorophyll a — Hovmöller depth profile at CAB (single row, 3 panels) ──
# Mirrors the DO depth-profile layout: (a) Baseline | (b) FPV | (c) Δ
# Same shared-y, two-colorbar arrangement.

def chlfa_depth_matrix(his_run, loc_dict):
    """Build (n_times, n_layers) Chlfa matrix at CAB or BIS across active
    layers. Self-contained: reads Chlfa from .his directly, with fallback
    to ecotype reconstruction."""
    layers = [k for k in sorted(loc_dict.keys())
              if his_k_to_waq(k) in range(K_TOP, K_BOT + 1)]
    if not layers:
        return None, None, None
    depths = [LAYER_MID_DEPTHS[his_k_to_waq(k)] for k in layers]

    cols = []
    for k_label in layers:
        # Try Chlfa directly from .his
        si = his_run['idx'].get('Chlfa')
        if si is not None and k_label in loc_dict:
            ts = pd.Series(
                his_run['data'][:, loc_dict[k_label], si].astype(float),
                index=his_run['ts_index']).resample('D').mean()
        else:
            # Fallback: sum all algae ecotypes × ChlaCAlg
            algae_subs = [s for s in his_run['substances']
                          if s.endswith(('_E', '_N', '_P', '_F'))
                          or s == 'FFLAGELA']
            ts = None
            for sp in algae_subs:
                sp_si = his_run['idx'].get(sp)
                if sp_si is None or k_label not in loc_dict:
                    continue
                contrib = pd.Series(
                    his_run['data'][:, loc_dict[k_label], sp_si].astype(float),
                    index=his_run['ts_index']
                ).resample('D').mean() * ChlaCAlg * 1000
                ts = contrib if ts is None else ts.add(contrib, fill_value=0)
            if ts is None:
                ts = pd.Series(dtype=float)
        cols.append(ts)

    if not cols or all(c.empty for c in cols):
        return None, None, None
    df = pd.concat(cols, axis=1)
    df.columns = layers
    return df.index, depths, df.values


# Build matrices for CAB only
ts_CAB_B, dep_CAB_B, chla_CAB_B = chlfa_depth_matrix(RB, cab_B)
ts_CAB_F, dep_CAB_F, chla_CAB_F = chlfa_depth_matrix(RF, cab_F)

if chla_CAB_B is None:
    print('Could not build Chlfa depth matrix at CAB — check that CAB layer '
          'monitors are present in .his.')
else:
    # Shared colour scale for absolute panels
    all_abs = np.concatenate([
        chla_CAB_B[~np.isnan(chla_CAB_B)],
        chla_CAB_F[~np.isnan(chla_CAB_F)],
    ])
    vmin, vmax = np.nanpercentile(all_abs, [2, 98])

    # Difference matrix on common times
    common = pd.DatetimeIndex(ts_CAB_B).intersection(pd.DatetimeIndex(ts_CAB_F))
    ib = [i for i, t in enumerate(ts_CAB_B) if t in common]
    i_f = [i for i, t in enumerate(ts_CAB_F) if t in common]
    diff_CAB = chla_CAB_F[i_f] - chla_CAB_B[ib]
    dvalid = diff_CAB[~np.isnan(diff_CAB)]
    dlim = np.nanpercentile(np.abs(dvalid), 98) if dvalid.size else 1.0

    # ── 1 row × 3 cols (CAB only) ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharey=True)

    # (a) Baseline
    pc_abs = axes[0].pcolormesh(ts_CAB_B, dep_CAB_B, chla_CAB_B.T,
                                  cmap='YlGn', vmin=vmin, vmax=vmax,
                                  shading='nearest')
    axes[0].set_ylabel('Depth (m)', fontsize=9)
    axes[0].text(0.02, 0.08, '(a) Baseline', transform=axes[0].transAxes,
                  fontsize=9, ha='left', va='top',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))

    # (b) FPV
    axes[1].pcolormesh(ts_CAB_F, dep_CAB_F, chla_CAB_F.T,
                        cmap='YlGn', vmin=vmin, vmax=vmax,
                        shading='nearest')
    axes[1].text(0.02, 0.08, '(b) FPV', transform=axes[1].transAxes,
                  fontsize=9, ha='left', va='top',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))

    # (c) Δ (FPV − Baseline)
    pc_diff = axes[2].pcolormesh(common, dep_CAB_B, diff_CAB.T,
                                   cmap='RdBu_r', vmin=-dlim, vmax=dlim,
                                   shading='nearest')
    axes[2].text(0.02, 0.08, '(c) Δ (FPV − Baseline)',
                  transform=axes[2].transAxes, fontsize=9, ha='left', va='top',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))

    # Common formatting
    for ax in axes:
        ax.set_xlim(xlim)
        ax.xaxis.set_major_formatter(xfmt)
        ax.xaxis.set_major_locator(xloc)
        ax.tick_params(labelsize=8)
    axes[0].invert_yaxis()

    # Two right-side colorbars (matching the DO plot)
    fig.subplots_adjust(left=0.06, right=0.86, top=0.95, bottom=0.18,
                        wspace=0.05)
    cbar_ax_abs  = fig.add_axes([0.87, 0.18, 0.012, 0.77])
    fig.colorbar(pc_abs, cax=cbar_ax_abs,
                  label=label_for('Chlfa') if 'label_for' in dir()
                        else 'Chlorophyll a (µg/L)')
    cbar_ax_diff = fig.add_axes([0.94, 0.18, 0.012, 0.77])
    fig.colorbar(pc_diff, cax=cbar_ax_diff,
                  label='Δ Chlorophyll a (µg/L)\n(FPV − Baseline)')

    plt.savefig(os.path.join(RESULTS_DIR, 'Chla_depth_profiles_CAB.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    # ── Summary stats ──────────────────────────────────────────────────────
    print('\nChla depth profile summary at CAB (Apr–Aug 2023):')
    print(f'  {"Scenario":<10} {"Surface mean":>14} '
          f'{"Bottom mean":>14} {"Column mean":>14}')
    print('  ' + '-' * 60)
    for scen_name, mat in [('Baseline', chla_CAB_B), ('FPV', chla_CAB_F)]:
        surf_mean = np.nanmean(mat[:, 0])
        bot_mean  = np.nanmean(mat[:, -1])
        col_mean  = np.nanmean(mat)
        print(f'  {scen_name:<10} {surf_mean:>14.3f} '
              f'{bot_mean:>14.3f} {col_mean:>14.3f}')

<Figure size 1000x300 with 5 Axes>


Chla depth profile summary at CAB (Apr–Aug 2023):
  Scenario     Surface mean    Bottom mean    Column mean
  ------------------------------------------------------------
  Baseline            8.779          0.406          2.678
  FPV                17.317          0.663          5.516


In [54]:
# ─────────────────────────────────────────────────────────────────────────
# COMBINED FIGURE: Chla depth profile (top) + ecotype biomass stack (bottom)
# ─────────────────────────────────────────────────────────────────────────
# DATA SOURCE — both rows at the CAB monitor point (single column):
#   Top row:    .his file, all K layers at CAB (8..16) — depth profile
#   Bottom row: .his file, CAB at K_label=8 (= WAQ K_TOP=7) — surface only
# ─────────────────────────────────────────────────────────────────────────

import matplotlib.gridspec as gridspec


# ── Ecotype biomass at CAB surface — built from .his ────────────────────
ECOTYPE_SUFFIXES = {
    'E (light-adapted)': '_E',
    'N (N-adapted)':     '_N',
    'P (P-adapted)':     '_P',
}
ECOTYPE_COLOURS = {
    'E (light-adapted)': '#2ca02c',
    'N (N-adapted)':     '#1f77b4',
    'P (P-adapted)':     '#d62728',
}
EXTRA_E_SPECIES = ['FFLAGELA']


def sum_ecotype_at_cab(R, suffix, loc_dict, k_label=8):
    species = [s for s in R['substances'] if s.endswith(suffix)]
    if suffix == '_E':
        species += [s for s in EXTRA_E_SPECIES if s in R['substances']]
    if not species:
        return pd.Series(dtype=float)
    total = None
    for sp in species:
        si = R['idx'].get(sp)
        if si is None or k_label not in loc_dict:
            continue
        ts = pd.Series(R['data'][:, loc_dict[k_label], si].astype(float),
                        index=R['ts_index']).resample('D').mean()
        total = ts if total is None else total.add(ts, fill_value=0)
    if total is None:
        return pd.Series(dtype=float)
    return total


ecotype_biomass = {}
for scen_name, R, loc_dict in [('Baseline', RB, cab_B), ('FPV', RF, cab_F)]:
    ecotype_biomass[scen_name] = {}
    for label, suffix in ECOTYPE_SUFFIXES.items():
        ecotype_biomass[scen_name][label] = sum_ecotype_at_cab(R, suffix, loc_dict)


# ── Chla scale & diff ───────────────────────────────────────────────────
all_abs = np.concatenate([
    chla_CAB_B[~np.isnan(chla_CAB_B)],
    chla_CAB_F[~np.isnan(chla_CAB_F)],
])
vmin, vmax = np.nanpercentile(all_abs, [2, 98])

common = pd.DatetimeIndex(ts_CAB_B).intersection(pd.DatetimeIndex(ts_CAB_F))
ib = [i for i, t in enumerate(ts_CAB_B) if t in common]
i_f = [i for i, t in enumerate(ts_CAB_F) if t in common]
diff_CAB = chla_CAB_F[i_f] - chla_CAB_B[ib]
dvalid = diff_CAB[~np.isnan(diff_CAB)]
dlim = np.nanpercentile(np.abs(dvalid), 98) if dvalid.size else 1.0


# ── Build figure with GridSpec — colorbars as ACTUAL gridspec cells ─────
# Top row: 5 cells = [Baseline | FPV | Δ | cbar_abs | cbar_diff]
# Bottom row: 5 cells = [Baseline | FPV | Δ | empty | empty]
# This guarantees column alignment across rows.

fig = plt.figure(figsize=(12, 7))
# 5 columns: [Baseline | FPV | cbar_abs | Δ | cbar_diff]
# Colorbar for abs (YlGn) sits between (b) FPV and (c) Δ.
# Colorbar for diff (RdBu_r) sits to the right of (c) Δ.
# Bottom row uses gs[1, 0], gs[1, 1], and gs[1, 3] — skipping the colorbar
# columns to preserve column alignment with the top row's data panels.
gs = gridspec.GridSpec(2, 5, figure=fig,
                       left=0.06, right=0.96, top=0.95, bottom=0.10,
                       hspace=0.32, wspace=0.30,
                       width_ratios=[1, 1, 0.05, 1, 0.05])

# Top row: data panels at cols 0, 1, 3; colorbars at cols 2 and 4
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1], sharey=ax_a)
cbar_ax_abs = fig.add_subplot(gs[0, 2])
ax_c = fig.add_subplot(gs[0, 3], sharey=ax_a)
cbar_ax_diff = fig.add_subplot(gs[0, 4])

plt.setp(ax_b.get_yticklabels(), visible=False)
plt.setp(ax_c.get_yticklabels(), visible=False)

pc_abs = ax_a.pcolormesh(ts_CAB_B, dep_CAB_B, chla_CAB_B.T,
                          cmap='YlGn', vmin=vmin, vmax=vmax, shading='nearest')
ax_a.set_ylabel('Depth (m)', fontsize=9)
ax_a.text(0.02, 0.08, '(a) Baseline', transform=ax_a.transAxes,
           fontsize=9, ha='left', va='top',
           bbox=dict(facecolor='white', edgecolor='none', pad=2.0, alpha=0.85))

ax_b.pcolormesh(ts_CAB_F, dep_CAB_F, chla_CAB_F.T,
                 cmap='YlGn', vmin=vmin, vmax=vmax, shading='nearest')
ax_b.text(0.02, 0.08, '(b) FPV', transform=ax_b.transAxes,
           fontsize=9, ha='left', va='top',
           bbox=dict(facecolor='white', edgecolor='none', pad=2.0, alpha=0.85))

pc_diff = ax_c.pcolormesh(common, dep_CAB_B, diff_CAB.T,
                           cmap='RdBu_r', vmin=-dlim, vmax=dlim,
                           shading='nearest')
ax_c.text(0.02, 0.08, '(c) Δ (FPV − Baseline)',
           transform=ax_c.transAxes, fontsize=9, ha='left', va='top',
           bbox=dict(facecolor='white', edgecolor='none', pad=2.0, alpha=0.85))

ax_a.invert_yaxis()
for ax in [ax_a, ax_b, ax_c]:
    ax.set_xlim(xlim)
    ax.xaxis.set_major_formatter(xfmt)
    ax.xaxis.set_major_locator(xloc)
    ax.tick_params(labelsize=8)

# Colorbars now inside gridspec → align perfectly with rows
fig.colorbar(pc_abs, cax=cbar_ax_abs, label='Chlorophyll a (µg/L)')
fig.colorbar(pc_diff, cax=cbar_ax_diff,
              label='Δ Chlorophyll a (µg/L)\n(FPV − Baseline)')

# Bottom row: skip cols 2 and 4 (those are the top row's colorbar columns)
ax_d = fig.add_subplot(gs[1, 0])
ax_e = fig.add_subplot(gs[1, 1], sharey=ax_d)
ax_f = fig.add_subplot(gs[1, 3])   # col 3 to align with (c)
# Cells [1, 3] and [1, 4] remain empty → align with top row's colorbars
plt.setp(ax_e.get_yticklabels(), visible=False)

stack_b = []
labels = []
for label in ECOTYPE_SUFFIXES.keys():
    series = ecotype_biomass['Baseline'][label]
    if series.empty:
        continue
    stack_b.append(series.values)
    labels.append(label)
times_b = ecotype_biomass['Baseline'][labels[0]].index
colours = [ECOTYPE_COLOURS[lab] for lab in labels]
ax_d.stackplot(times_b, *stack_b, labels=labels, colors=colours, alpha=0.85)
ax_d.set_ylabel('Biomass (gC/m³)', fontsize=9)
ax_d.text(0.02, 0.97, '(d) Baseline', transform=ax_d.transAxes,
           fontsize=9, ha='left', va='top',
           bbox=dict(facecolor='white', edgecolor='none', pad=2.0, alpha=0.85))
ax_d.legend(fontsize=8, loc='upper left', bbox_to_anchor=(0.0, 0.93))
ax_d.grid(True, alpha=0.3)

stack_f = []
for label in ECOTYPE_SUFFIXES.keys():
    series = ecotype_biomass['FPV'][label]
    if series.empty:
        continue
    stack_f.append(series.values)
times_f = ecotype_biomass['FPV'][labels[0]].index
ax_e.stackplot(times_f, *stack_f, labels=labels, colors=colours, alpha=0.85)
ax_e.text(0.02, 0.97, '(e) FPV', transform=ax_e.transAxes,
           fontsize=9, ha='left', va='top',
           bbox=dict(facecolor='white', edgecolor='none', pad=2.0, alpha=0.85))
ax_e.grid(True, alpha=0.3)

common_idx = ecotype_biomass['Baseline'][labels[0]].index.intersection(
    ecotype_biomass['FPV'][labels[0]].index)
for label in labels:
    base = ecotype_biomass['Baseline'][label].loc[common_idx]
    fpv  = ecotype_biomass['FPV'][label].loc[common_idx]
    diff = fpv - base
    ax_f.plot(diff.index, diff.values, color=ECOTYPE_COLOURS[label],
              lw=1.3, label=label)
ax_f.axhline(0, color='k', lw=0.7, ls='--', alpha=0.5)
ax_f.set_ylabel('Δ FPV − Baseline (gC/m³)', fontsize=9)
ax_f.text(0.02, 0.97, '(f) Δ ecotype biomass (FPV − Baseline)',
           transform=ax_f.transAxes, fontsize=9, ha='left', va='top',
           bbox=dict(facecolor='white', edgecolor='none', pad=2.0, alpha=0.85))
ax_f.grid(True, alpha=0.3)

for ax in [ax_d, ax_e, ax_f]:
    ax.set_xlim(pd.Timestamp('2023-04-01'), pd.Timestamp('2023-08-31'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.tick_params(labelsize=8)


plt.savefig(os.path.join(RESULTS_DIR,
                          'Combined_Chla_and_ecotype_biomass_CAB.png'),
            dpi=150, bbox_inches='tight')
plt.show()


print('\nEcotype biomass summary at CAB surface (Apr-Aug 2023):')
print(f'  {"Ecotype":<22} {"Baseline mean":>14} {"FPV mean":>14} '
      f'{"Δ abs":>10} {"Δ %":>10}')
print('  ' + '-' * 80)
for label in labels:
    bm = ecotype_biomass['Baseline'][label].mean()
    fm = ecotype_biomass['FPV'][label].mean()
    d_abs = fm - bm
    d_pct = (d_abs / bm * 100) if bm != 0 else float('nan')
    print(f'  {label:<22} {bm:>14.4f} {fm:>14.4f} '
          f'{d_abs:>+10.4f} {d_pct:>+9.2f}%')

<Figure size 1200x700 with 8 Axes>


Ecotype biomass summary at CAB surface (Apr-Aug 2023):
  Ecotype                 Baseline mean       FPV mean      Δ abs        Δ %
  --------------------------------------------------------------------------------
  E (light-adapted)              0.1512         0.0856    -0.0656    -43.36%
  N (N-adapted)                  0.0000         0.0001    +0.0001   +154.06%
  P (P-adapted)                  0.1533         0.5802    +0.4269   +278.56%


## Section 7 — Seasonal difference boxplots

In [24]:
# ── Local helper definitions (self-contained for this cell) ─────────────
def chla_from_his(his_run, cab_dict, k_label=8):
    """Chlfa from .his at CAB(k_label). Fallback to stoichiometric estimate
    if Chlfa absent from .his."""
    si = his_run['idx'].get('Chlfa')
    if si is not None and k_label in cab_dict:
        ts = pd.Series(his_run['data'][:, cab_dict[k_label], si].astype(float),
                       index=his_run['ts_index'])
        return ts.resample('D').mean()
    # Fallback: sum all algae ecotypes × ChlaCAlg
    algae_subs = [s for s in his_run['substances']
                  if s.endswith(('_E', '_N', '_P', '_F'))
                  or s == 'FFLAGELA']
    cols = []
    for sp in algae_subs:
        sp_si = his_run['idx'].get(sp)
        if sp_si is None or k_label not in cab_dict:
            continue
        cols.append(pd.Series(
            his_run['data'][:, cab_dict[k_label], sp_si].astype(float),
            index=his_run['ts_index']).resample('D').mean())
    return sum(cols) * ChlaCAlg * 1000 if cols else pd.Series(dtype=float)


def secchi_his(his_run, cab_dict, k_label=8):
    """SecchiDept from .his — preferred. Falls back to 1.7/ExtVl proxy."""
    si = his_run['idx'].get('SecchiDept')
    if si is not None and k_label in cab_dict:
        ts = pd.Series(his_run['data'][:, cab_dict[k_label], si].astype(float),
                       index=his_run['ts_index'])
        return ts.resample('D').mean()
    si = his_run['idx'].get('ExtVl')
    if si is not None and k_label in cab_dict:
        ev = his_run['data'][:, cab_dict[k_label], si].astype(float)
        out = np.full_like(ev, np.nan, dtype=float)
        np.divide(1.7, ev, out=out, where=ev > 0)
        return pd.Series(out, index=his_run['ts_index']).resample('D').mean()
    return pd.Series(dtype=float)

In [25]:
# Build month windows automatically across SIM_START..SIM_END
def _build_month_windows(start, end):
    """Return list of (label, t_start, t_end) tuples — one per month."""
    months = pd.date_range(start=pd.Timestamp(start),
                           end=pd.Timestamp(end), freq='MS')
    windows = []
    for m in months:
        label = m.strftime('%b\n%Y')
        t0 = m
        t1 = (m + pd.offsets.MonthEnd(0))
        windows.append((label, t0.strftime('%Y-%m-%d'),
                        t1.strftime('%Y-%m-%d')))
    return windows


month_windows = _build_month_windows(SIM_START, SIM_END)
month_colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(month_windows)))


def do_07m(ds, his_run, cab_dict):
    s8 = get_series(ds, his_run, 'OXY', 8, m=cab_m, n=cab_n)
    s9 = get_series(ds, his_run, 'OXY', 9, m=cab_m, n=cab_n)
    return s8

var_funcs = [
    ('DO (g/m³)',    lambda ds, his, c: do_07m(ds, his, c)),
    ('Chl a (µg/L)', lambda ds, his, c: chla_from_his(his, c, k_label=8)),
    ('NO3 (gN/m³)',  lambda ds, his, c: depth_avg(ds, his, 'NO3', c,
                                                    m=cab_m, n=cab_n)),
    ('PO4 (gP/m³)',  lambda ds, his, c: depth_avg(ds, his, 'PO4', c,
                                                    m=cab_m, n=cab_n)),
    ('Si (gSi/m³)',  lambda ds, his, c: depth_avg(ds, his, 'Si', c,
                                                    m=cab_m, n=cab_n)),
    ('Secchi (m)',   lambda ds, his, c: secchi_his(his, c, k_label=8)),
]

# Multi-row layout: 2 rows × 3 cols for 6 variables
nrows = 2
ncols = 3
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows),
                          squeeze=False)
axes_flat = axes.flatten()

panel_letters = list('abcdefghij')

for ax, (var_name, func), letter in zip(axes_flat, var_funcs, panel_letters):
    vb = func(ds_B, RB, cab_B); vf = func(ds_F, RF, cab_F)
    if vb.empty or vf.empty:
        ax.set_title(f'{var_name}\n(no data)', fontsize=9)
        continue
    common = vb.index.intersection(vf.index)
    diff = vf.loc[common] - vb.loc[common]
    data_list = [diff.loc[t0:t1].dropna().values
                 for _, t0, t1 in month_windows]
    labels    = [lbl for lbl, _, _ in month_windows]

    bp = ax.boxplot(data_list, tick_labels=labels,
                    patch_artist=True, widths=0.55)
    for patch, c in zip(bp['boxes'], month_colors):
        patch.set_facecolor(c); patch.set_alpha(0.7)
    ax.axhline(0, color='k', lw=1.0, ls='--')
    ax.set_ylabel('Δ (FPV − Baseline)', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.3, axis='y')

    # Panel letter + variable name as annotation (no title)
    ax.text(0.02, 0.97, f'({letter}) {var_name}',
            transform=ax.transAxes,
            fontsize=10, ha='left', va='top',
            bbox=dict(facecolor='white', edgecolor='none',
                      pad=2.0, alpha=0.85))

# Hide unused panels (if number of variables < nrows*ncols)
for j in range(len(var_funcs), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Monthly FPV impact — CAB (FPV minus Baseline)', fontsize=11,
             y=1.00)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'Monthly_differences_CAB.png'),
            dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1200x700 with 6 Axes>

## Section 8 — Performance summary table

In [26]:
# Build month-by-month comparison columns, plus an "Apr-Aug" total at the end
def _build_month_columns(start, end):
    """Return ordered dict {label: (t_start, t_end)} for each month from
    start to end (inclusive), plus a final 'Apr-Aug' total column."""
    months = pd.date_range(start=pd.Timestamp(start),
                           end=pd.Timestamp(end), freq='MS')
    cols = {}
    for m in months:
        label = m.strftime('%b %y')
        t0 = m
        t1 = (m + pd.offsets.MonthEnd(0))
        cols[label] = (t0, t1)
    cols['Apr-Aug'] = (start, end)
    return cols

months_tbl = _build_month_columns(SIM_START, SIM_END)


# ── Helpers for .his-only variables ──────────────────────────────────────
def radave_k(his_run, cab_dict, k_label=8):
    si = his_run['idx'].get('RadAve')
    if si is None or k_label not in cab_dict:
        return pd.Series(dtype=float)
    ts = pd.Series(his_run['data'][:, cab_dict[k_label], si].astype(float),
                   index=his_run['ts_index'])
    return ts.resample('D').mean()

def vwind_k(his_run, cab_dict, k_label=8):
    si = his_run['idx'].get('VWind')
    if si is None or k_label not in cab_dict:
        return pd.Series(dtype=float)
    ts = pd.Series(his_run['data'][:, cab_dict[k_label], si].astype(float),
                   index=his_run['ts_index'])
    return ts.resample('D').mean()


# ── Variable definitions: (name, unit, surface_func, depthavg_func) ──────
# Each variable produces two rows: one for the surface (K=7 / .his label 8)
# and one for the depth-averaged value. None in a slot means "no row of
# that kind for this variable."
var_tbl = [
    ('DO',     'g/m3',
        lambda ds, h, c: get_series(ds, h, 'OXY', 8, m=cab_m, n=cab_n),
        lambda ds, h, c: depth_avg(ds, h, 'OXY', c, m=cab_m, n=cab_n)),
    ('Chl a',  'ug/L',
        lambda ds, h, c: chla_from_his(h, c, k_label=8),
        None),
    ('NO3',    'gN/m3',
        lambda ds, h, c: get_series(ds, h, 'NO3', 8, m=cab_m, n=cab_n),
        lambda ds, h, c: depth_avg(ds, h, 'NO3', c, m=cab_m, n=cab_n)),
    ('PO4',    'gP/m3',
        lambda ds, h, c: get_series(ds, h, 'PO4', 8, m=cab_m, n=cab_n),
        lambda ds, h, c: depth_avg(ds, h, 'PO4', c, m=cab_m, n=cab_n)),
    ('Si',     'gSi/m3',
        lambda ds, h, c: get_series(ds, h, 'Si', 8, m=cab_m, n=cab_n),
        lambda ds, h, c: depth_avg(ds, h, 'Si', c, m=cab_m, n=cab_n)),
    ('Secchi', 'm',
        lambda ds, h, c: secchi_his(h, c, k_label=8),
        None),
    ('RadAve', 'W/m2',
        lambda ds, h, c: radave_k(h, c, k_label=8),
        None),
    ('VWind',  'm/s',
        lambda ds, h, c: vwind_k(h, c, k_label=8),
        None),
]


# Column widths
col_w = 10
n_cols = len(months_tbl)
total_w = 13 + 8 + 11 + 2 + 9 + n_cols * (col_w + 2)

print('\n' + '=' * total_w)
print('FPV SCENARIO IMPACT SUMMARY — CAB (monthly)')
print('=' * total_w)
hdr = f'{"Variable":<13} {"Unit":<8} {"Layer":<10}  {"":<9}'
for s in months_tbl:
    hdr += f'  {s:>{col_w}}'
print(hdr)
print('-' * total_w)


def _print_rows(var_name, unit, layer_label, func, first_block):
    """Print Baseline/FPV/Delta rows AND record them for CSV export."""
    vb = func(ds_B, RB, cab_B); vf = func(ds_F, RF, cab_F)
    if vb.empty or vf.empty:
        prefix_var = var_name if first_block else ''
        print(f'{prefix_var:<13} {unit:<8} {layer_label:<10}  (no data)')
        return
    common = vb.index.intersection(vf.index)
    diff = vf.loc[common] - vb.loc[common]
    for i, (row_label, row_data) in enumerate(
            [('Baseline', vb), ('FPV', vf), ('Delta', diff)]):
        prefix_var = var_name if (first_block and i == 0) else ''
        prefix_unit = unit if (first_block and i == 0) else ''
        prefix_layer = layer_label if i == 0 else ''
        line = f'{prefix_var:<13} {prefix_unit:<8} {prefix_layer:<10}  {row_label:<9}'

        # For CSV: always carry full identifiers (no leading blanks)
        csv_row = {'Variable': var_name, 'Unit': unit,
                   'Layer': layer_label, 'Scenario': row_label}
        for s_label, (t0, t1) in months_tbl.items():
            mean = row_data.loc[t0:t1].mean()
            sign = '+' if (row_label == 'Delta' and mean >= 0) else ''
            line += f'  {sign}{mean:>{col_w - 1}.3f}'
            csv_row[s_label] = mean
        print(line)
        csv_rows.append(csv_row)

csv_rows = []   # accumulate rows here for export

for var_name, unit, surf_func, davg_func in var_tbl:
    # Surface row
    if surf_func is not None:
        _print_rows(var_name, unit, 'Surface', surf_func, first_block=True)
    # Depth-averaged row (only if defined)
    if davg_func is not None:
        _print_rows(var_name, unit, 'Depth-avg', davg_func, first_block=(surf_func is None))
    print()

print('=' * total_w)
csv_path = os.path.join(RESULTS_DIR, 'FPV_impact_summary_CAB_monthly.csv')
pd.DataFrame(csv_rows).to_csv(csv_path, index=False)
print(f'\nTable exported to: {csv_path}')


FPV SCENARIO IMPACT SUMMARY — CAB (monthly)
Variable      Unit     Layer                      Apr 23      May 23      Jun 23      Jul 23      Aug 23     Apr-Aug
-------------------------------------------------------------------------------------------------------------------
DO            g/m3     Surface     Baseline       9.883      9.125      9.268      8.556      7.860      8.930
                                   FPV           10.209      9.539      9.968      9.285      9.018      9.597
                                   Delta      +    0.326  +    0.414  +    0.700  +    0.729  +    1.158  +    0.667
                       Depth-avg   Baseline       9.910      9.091      8.299      7.054      7.381      8.337
                                   FPV           10.211      9.346      8.274      6.378      6.310      8.089
                                   Delta      +    0.300  +    0.255     -0.024     -0.676     -1.071     -0.248

Chl a         ug/L     Surface     Baseline    

In [31]:
def _find_strongest_attenuation_cell(path_b, path_f, n_seg=N_SEG,
                                       sample_t=range(0, 200, 5),
                                       segnum=None, new_to_old=None,
                                       target_k=K_TOP):
    """Find the compact segment with strongest attenuation, restricted to
    cells at K=target_k (surface within the WAQ active domain)."""
    rec_size = 4 + n_seg * 4
    diff_sum = np.zeros(n_seg, dtype=np.float64)
    with open(path_b, 'rb') as fb, open(path_f, 'rb') as ff:
        for t_idx in sample_t:
            fb.seek(t_idx * rec_size + 4)
            ff.seek(t_idx * rec_size + 4)
            row_b = np.frombuffer(fb.read(n_seg * 4), dtype='<f4').copy()
            row_f = np.frombuffer(ff.read(n_seg * 4), dtype='<f4').copy()
            diff_sum += np.abs(row_b - row_f)

    # Build set of flow_segs that exist at K=target_k in segnum
    flow_segs_at_target_k = set()
    for m_ in range(segnum.shape[0]):
        for n_ in range(segnum.shape[1]):
            v = segnum[m_, n_, target_k]
            if not np.isnan(v):
                flow_segs_at_target_k.add(int(v))

    print(f'  Found {len(flow_segs_at_target_k)} cells at K={target_k} in segnum')

    # Filter: keep only compact segments whose flow_seg is at target_k
    is_at_target_k = np.zeros(n_seg, dtype=bool)
    for compact_idx in range(n_seg):
        flow_seg = new_to_old[compact_idx] + 1   # 1-based
        if flow_seg in flow_segs_at_target_k:
            is_at_target_k[compact_idx] = True

    print(f'  {is_at_target_k.sum()} of {n_seg} compact segments at K={target_k}')

    # Mask out cells not at target_k, then find the max
    diff_sum_masked = diff_sum.copy()
    diff_sum_masked[~is_at_target_k] = 0
    return int(np.argmax(diff_sum_masked)), diff_sum_masked


# Call with explicit segnum and new_to_old
target_compact, _ = _find_strongest_attenuation_cell(
    RAD_BIN['Baseline'], RAD_BIN['FPV'],
    segnum=segnum, new_to_old=new_to_old, target_k=K_TOP)

target_flow = new_to_old[target_compact] + 1
print(f'\nResult: compact={target_compact}, flow_seg={target_flow}')

# Now reverse-lookup the (M, N, K) — should be at K=K_TOP
target_mnk = None
for m in range(segnum.shape[0]):
    for n in range(segnum.shape[1]):
        v = segnum[m, n, K_TOP]   # <— explicitly look at K=K_TOP only
        if not np.isnan(v) and int(v) == target_flow:
            target_mnk = (m, n, K_TOP)
            break
    if target_mnk:
        break

print(f'Target cell: (M={target_mnk[0]}, N={target_mnk[1]}, K={target_mnk[2]})')


# ── Spatial plot: surface grid with the under-panel target cell marked ──
# Shows the model grid (Baseline OXY surface, daily mean for the analysis
# midpoint) with the target FPV-attenuation cell highlighted.

import matplotlib.patches as mpatches

# Use the midpoint of the analysis window for the background field
mid_date = SIM_START + (pd.Timestamp(SIM_END) - pd.Timestamp(SIM_START)) / 2
mid_date = pd.Timestamp(mid_date.date())

# Baseline surface field for background — using OXY for visual reference
bg_field = ds_B['OXY'].sel(K=K_TOP).sel(time=mid_date, method='nearest').values

# Reuse the perimeter mask already built in the previous cell
fpv_mask_mn = perim_da.values

fig, ax = plt.subplots(figsize=(7, 6))

# Background: OXY surface as the grid context (greyscale)
pc = ax.pcolormesh(np.ma.masked_invalid(bg_field.T),
                   cmap='Greys', alpha=0.5, shading='nearest')
plt.colorbar(pc, ax=ax, label='Baseline DO at surface (g/m³) — context only',
             shrink=0.7)

# FPV footprint outline (semi-transparent yellow fill)
ax.pcolormesh(np.ma.masked_where(~fpv_mask_mn, fpv_mask_mn).T,
              cmap='YlOrBr', alpha=0.35, shading='nearest')

# Mark the target cell (the under-panel cell we analyse for radiation)
tm, tn, tk = target_mnk
ax.plot(tm + 0.5, tn + 0.5, marker='o', markersize=14,
        markerfacecolor='red', markeredgecolor='black', markeredgewidth=1.5,
        label=f'Target cell (M={tm}, N={tn}, K={tk})')

# Mark CAB for reference (it's outside the FPV footprint)
ax.plot(cab_m + 0.5, cab_n + 0.5, marker='s', markersize=12,
        markerfacecolor='cyan', markeredgecolor='black', markeredgewidth=1.5,
        label=f'CAB monitor (M={cab_m}, N={cab_n})')

# Mark BIS if known
if 'bis_m' in dir() and 'bis_n' in dir():
    ax.plot(bis_m + 0.5, bis_n + 0.5, marker='s', markersize=12,
            markerfacecolor='magenta', markeredgecolor='black', markeredgewidth=1.5,
            label=f'BIS monitor (M={bis_m}, N={bis_n})')

# Custom legend handles for the FPV footprint patch
fpv_patch = mpatches.Patch(color='#FF9F40', alpha=0.5,
                            label=f'FPV array footprint ({fpv_mask_mn.sum()} cells)')
handles, labels = ax.get_legend_handles_labels()
handles.append(fpv_patch); labels.append(fpv_patch.get_label())
ax.legend(handles=handles, labels=labels, fontsize=9,
          loc='upper right', framealpha=0.9)

ax.set_xlabel('M (grid index)', fontsize=10)
ax.set_ylabel('N (grid index)', fontsize=10)
ax.set_title('Reservoir grid — FPV footprint and analysis locations\n'
             f'Target cell at compact={target_compact} ('
             f'strongest FPV attenuation signal)',
             fontsize=10)
ax.set_aspect('equal')
ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'Grid_FPV_locations.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print(f'\nLocations marked on grid:')
print(f'  Target cell:  (M={tm}, N={tn}, K={tk})  '
      f'compact={target_compact}')
print(f'  CAB monitor:  (M={cab_m}, N={cab_n})')
if 'bis_m' in dir():
    print(f'  BIS monitor:  (M={bis_m}, N={bis_n})')
print(f'  FPV array:    {fpv_mask_mn.sum()} surface cells')

  Found 1744 cells at K=7 in segnum
  1357 of 10999 compact segments at K=7

Result: compact=446, flow_seg=12249
Target cell: (M=3, N=37, K=7)


<Figure size 700x600 with 2 Axes>


Locations marked on grid:
  Target cell:  (M=3, N=37, K=7)  compact=446
  CAB monitor:  (M=23, N=36)
  BIS monitor:  (M=37, N=12)
  FPV array:    469 surface cells
